# 0. Эксперимент 3 - HMM только для токенов

**Гипотеза:** отделить изменение кандидатов CoDD от прямой замены оценок уверенности. Кандидаты берутся после совместного выбора скрытых состояний HMM; уверенность вычисляется по исходным логитам LLaDA. Это не независимое семплирование маргиналов и не обучение новой модели.

Основной режим: `cand_from="pc"` - источник токенов после HMM, `order_from="base"` - источник уверенности до HMM. При `low_confidence` оцениваем вероятность **кандидата CoDD** под LLaDA, поэтому кандидат косвенно влияет на очередь. При `margin` оценка не зависит от выбранного кандидата. Изменившийся контекст на последующих шагах всё равно меняет обе траектории.

**Статус поставки: НЕ ПРОВЕРЕНО на GPU.** Ячейки не выполнялись при создании; веса, датасеты и зависимости не скачивались. Точность, latency, совместимость DataSphere и воспроизводимость PyJuice пока НЕ ПРОВЕРЕНЫ. Никаких результатов ниже заранее не подставлено.

Проверено чтением исходников форка:
- `CoddLlada.forward` вызывает LLaDA, затем `apply_pc_logits`. Последняя меняет логиты на месте, только в текущем блоке и при строгом условии доли масок `< pc_frac` (либо `>` при `reverse_frac=True`).
- `pc_temperature=0.2`, `pc_frac=0.5` соответствуют примеру README. По решению пользователя после сверки с `notebooks/exp1_llada_default.ipynb` установлены **`GEN_LENGTH=512` и `HMM_CKPT="il18/llada-gsm-pc"`**, вместо 256 и math-PC в исходном плане; gsm-PC присутствует в таблице README. Это **не** defaults: `CoDDConfig` содержит `0.1/0.7`, CLI - `0.7/0.3`. Здесь параметры всегда передаются явно; gsm-PC выбран явно по решению пользователя.
- `LladaEvalHarness._model_generate` задаёт 512 новых токенов. Здесь собственный цикл на 512 токенов, а подготовка промптов и проверка ответов остаются из встроенного harness.
- `add_gumbel_noise` используется как в оригинале, но этот эксперимент допускает только температуру 0. Режимы `random` и `entropy` предусмотрены для проверки формул, в основной прогон не включены.
- В корне нет отдельного requirements. `environment.yml`: Python 3.12.12, torch 2.9.1, transformers 4.57.3, pyjuice 2.4.1; встроенный harness - 0.4.10.dev0. Это сведения из файлов, не доказательство совместимости с GPU.

Все новые определения находятся в блокноте; существующие `.py` импортируются без изменений. Полный прогон: 12 конфигураций, всего 1200 ответов. Запускать ячейки последовательно, только в Linux-окружении DataSphere с NVIDIA CUDA.

Основной референс: `notebooks/exp1_llada_default.ipynb`. Согласованы длина, HMM ID, zero-shot/chat template, построение запросов, ID задач, хэши вопросов и промптов, поля результатов и шагового журнала. Отличия: HMM обязательна, три seed, две ветки логитов, диагностические метрики, более строгая проверка offline и транзакционное возобновление. Bootstrap здесь 2000 повторов/seed 20261023 по утверждённому плану; у exp1 - 10000/12345, поэтому границы CI могут немного отличаться.


In [ ]:
# 0.0. КОНФИГ - выполнить и локально, и на DataSphere
from pathlib import Path

MODEL = "GSAI-ML/LLaDA-8B-Instruct"
HMM_CKPT = "il18/llada-gsm-pc"
DATASET = "gsm8k"
SPLIT = "test"
N_TASKS = 100
GEN_LENGTH = 512
BLOCK_LENGTH = 32
TEMPERATURE = 0.0
NUM_STEPS_LIST = [64, 128]
REMASKING_LIST = ["low_confidence", "margin"]
PC_TEMPERATURE = 0.2
PC_FRAC = 0.5
REVERSE_FRAC = False
SEEDS = [0, 1, 2]

# Локально: заполните только ASSETS_DIR и включите RUN_LOCAL_DOWNLOAD.
# На DataSphere: ASSETS_DIR - путь к перенесённой папке; RUN_LOCAL_DOWNLOAD=False.
ASSETS_DIR = Path(r"../../exp3-assets")    # Path(r"C:\...\exp3-assets") локально; Path(r"../../exp3-assets") на DataSphere
RUN_LOCAL_DOWNLOAD = True # запускает ТОЛЬКО ячейку 0.1, без модели, CUDA и PyJuice
NOTEBOOK_RELATIVE_PATH = Path("notebooks/exp3_hmm_words_only.ipynb")
SOURCE_GIT_REVISION = "cb113a39eecea28591b164c705318db450b17c54" # базовая ревизия для ZIP без .git; изменения notebook фиксируются отдельно

# Заполните пути реального проекта DataSphere; существование проверяется ниже.
REPO_ROOT = None            # Path(".../CoDD-ablations") - checkout исходников
STORAGE_ROOT = None         # Path("...") - постоянное хранилище, не временный диск
OFFLINE = True              # True: запрещены обращения к Hugging Face
MODEL_LOCAL_DIR = None      # ручной путь; при ASSETS_DIR оставьте None (берётся ASSETS_DIR/model)
HMM_LOCAL_PATH = None       # ручной путь; при ASSETS_DIR оставьте None (hmm/model.jpc)
DATASET_LOCAL_DIR = None    # ручной путь; при ASSETS_DIR оставьте None (gsm8k)
HF_CACHE_DIR = None         # необязательный постоянный cache, используется online
MODEL_REVISION = "main"     # online: разрешается в commit SHA до загрузки
HMM_REVISION = "main"
DATASET_REVISION = "main"
LOCAL_MODEL_REVISION = None # если неизвестна: НЕ ПРОВЕРЕНО, вместо неё сохраняются SHA-256
LOCAL_HMM_REVISION = None
LOCAL_DATASET_REVISION = None

DEVICE = "cuda:0"
PRECISION = "bfloat16"
CFG_SCALE = 0.0
BATCH_SIZE = 1
NUM_FEWSHOT = 0
SYSTEM_INSTRUCTION = "You are a helpful assistant."
CAND_FROM = "pc"
ORDER_FROM = "base"
BOOTSTRAP_REPEATS = 2000
BOOTSTRAP_SEED = 20261023
CONFIDENCE_LEVEL = 0.95
SMOKE_TASKS = 3
SMOKE_GEN_LENGTH = 64
SMOKE_NUM_STEPS = 16
RUN_FULL = False            # включить только после успешной секции 5
BASELINE_RESULTS_DIR = None # None = STORAGE_ROOT/results/exp1_llada_default; используется только для сверки протокола
SCHEMA_VERSION = 1
EXPECTED_SOURCE_SHA256 = {
    "README.md": "84be531d3c16154a7fafb4a8d9393e92dbb35d43a680546d04bf2df0eb4ec2c9",
    "environment.yml": "8d9bdb8ea8e72f8c6ba74a0651a0d7450b15c3c93930701f0e8f6aead2927565",
    "codd/model_class.py": "c8abd5a4ef0edc78215602e811ec91fd7bcb965d4a85b7a2881e6e03108a8dfb",
    "eval/llada/codd_llada.py": "17c818d3c6eca68578c672a4a475eaa370ad1bbab8a9040590e81f9124ea16fb",
    "eval/llada/llada_generate.py": "6617851cd247e628839f0193764a594a7afccb8ecfa048a2af643be69d579c8a",
    "eval/llada/modeling_llada.py": "98bac7e53fef0bb7ca01e3716c11a7f710d183e10dbb9783b88db9dbba2e3766",
    "eval/llada/configuration_llada.py": "0fc0c048d25dfaa42f9657f202b96c41e9ed669099a2c72bc0fe0f9139ad78b2",
    "eval/eval.py": "45c4a5c9c91786b5e84ec1789e864b9f140e2aaaa7bc97c36c5875a70bebf583",
    "eval/harness.py": "dca146bb8e18c5b8c8474001ad7aa45d8ad8025701a88cb61ab8d248a4448ac9",
    "eval/utils.py": "4b0dfc11ee76abe8bea817f2ce4533f8b128f066eeb236d505c9f862d54b0562",
    "lm-evaluation-harness/pyproject.toml": "ee3c287541f1afe9f0a8558081051d81d383b2d811aec70993cf4f8cf7c97b17",
    "lm-evaluation-harness/lm_eval/tasks/gsm8k/gsm8k.yaml": "442b817bc83138ec89bd30ea9d6b4acac977dfc4653c177ad778f354c8543d29",
    "lm-evaluation-harness/lm_eval/api/task.py": "422f0dace38d28fd9dc5d320c33521b3fc3511a1378d180b39d5b5f836062f4a",
    "lm-evaluation-harness/lm_eval/api/metrics.py": "bf217f7dae049098959886592a059b5132faf45fb456139eb76b9a16dd4ba0a1",
    "lm-evaluation-harness/lm_eval/api/filter.py": "e174744a3ecaff78af34a60d6074bc8945801da50c6028d58f464cb2866c7dfe",
    "lm-evaluation-harness/lm_eval/filters/__init__.py": "b407c0dd4eb9fc9f9ae550a70122363d86aa7f9e32463f02d32e476694717c60",
    "lm-evaluation-harness/lm_eval/models/huggingface.py": "249b6089886a6942ccb332ed6fffc2cacdf1f8606be9d929eea3a91ebebb5c09",
    "lm-evaluation-harness/lm_eval/models/utils.py": "5ebf9350fe5a2479682526efb799633bfa078ba9b487ae028563b79bf5332eeb",
    "notebooks/exp1_llada_default.ipynb": "cb8102abeb6874b9f3e31ac35fef80da9c48f1c60f6d9269e277459600f34526"
}


## 0.1. Локальное скачивание без GPU - выполнить только на Windows-машине

**Локально выполните только две кодовые ячейки: `0.0. КОНФИГ`, затем `0.1. ЛОКАЛЬНОЕ СКАЧИВАНИЕ`.** В конфиге задайте `ASSETS_DIR = Path(r"C:\путь\exp3-assets")` (лучше вне репозитория, на диске с достаточным свободным местом) и `RUN_LOCAL_DOWNLOAD=True`. `REPO_ROOT`, `STORAGE_ROOT` и `OFFLINE` для этой ячейки не используются. Модель не загружается в память, PyTorch/CUDA/PyJuice не требуются.

Если нет библиотек, установите их в ядро этого локального Jupyter отдельной командой `%pip install "huggingface_hub==0.36.0" "datasets==4.4.1"`, затем при необходимости перезапустите ядро. Ячейка сама пакеты не устанавливает. Скачать файлы можно с CPU; зависимости Linux/CUDA для DataSphere этим не подготавливаются.

`ASSETS_DIR` получит `model/` (полный snapshot модели с токенизатором), `hmm/model.jpc`, `gsm8k/` (DatasetDict train/test) и `download_manifest.json` с точными ревизиями. Ревизии фиксируются до больших загрузок; повторный запуск продолжает подготовку тех же ревизий. `ready=true` записывается только после завершения всех трёх артефактов. `save_download_manifest(value)` атомарно сохраняет состояние; `value` - словарь ревизий, ID и статуса готовности. Свободное место проверяется по размерам файлов Hub с запасом 1 ГиБ для GSM8K/служебных файлов; дополнительная копия весов в основном HF-кэше не создаётся благодаря `local_dir`.

Справка API: [snapshot_download / hf_hub_download](https://huggingface.co/docs/huggingface_hub/v0.36.0/package_reference/file_download), [сохранение DatasetDict](https://huggingface.co/docs/datasets/v4.4.1/package_reference/main_classes#datasets.DatasetDict.save_to_disk). Выполнение скачивания и перенос ещё **НЕ ПРОВЕРЕНЫ** при подготовке блокнота.

**На DataSphere:** загрузите/распакуйте репозиторий целиком (включая `notebooks/exp1_llada_default.ipynb` и встроенный harness) и четыре элемента папки артефактов. Папки `.cache` и `.datasets_cache` переносить не нужно. В конфиге задайте `REPO_ROOT` (корень распакованного репозитория), `STORAGE_ROOT` (постоянное хранилище), `ASSETS_DIR` (папку перенесённых артефактов), оставьте `OFFLINE=True`, поставьте `RUN_LOCAL_DOWNLOAD=False`; ручные пути к модели/HMM/датасету оставьте `None`.

На новом GPU-ядре выполните конфиг, **пропустите скачивание 0.1**, затем выполните все кодовые ячейки с `1.1. ОКРУЖЕНИЕ DATASPHERE` до дымового теста включительно. После `SMOKE_PASSED=True` установите `RUN_FULL=True` отдельной строкой в начале ячейки раздела 6, сохраните блокнот и выполните эту ячейку, затем раздел 7. Повторно загружать модель или выполнять конфиг для переключения этого флага не нужно. Требование аппаратного bfloat16 сохраняется (T4 не подходит).

In [6]:
# 0.1. ЛОКАЛЬНОЕ СКАЧИВАНИЕ - после конфига; на DataSphere пропустить
if not RUN_LOCAL_DOWNLOAD:
    print("Скачивание пропущено: RUN_LOCAL_DOWNLOAD=False.")
else:
    import os
    import sys
    import json
    import shutil
    import tempfile
    import importlib.metadata

    assert ASSETS_DIR is not None, r"В конфиге задайте ASSETS_DIR = Path(r'C:\...\exp3-assets')."
    if any(name in sys.modules for name in ["huggingface_hub", "datasets"]) and (
        os.environ.get("HF_HUB_OFFLINE") == "1" or os.environ.get("HF_DATASETS_OFFLINE") == "1"
    ):
        raise RuntimeError("Начните локальное скачивание в новом ядре: библиотеки уже импортированы в offline-режиме.")
    os.environ["HF_HUB_OFFLINE"] = "0"
    os.environ["HF_DATASETS_OFFLINE"] = "0"
    from huggingface_hub import HfApi, snapshot_download, hf_hub_download
    import datasets

    assets_root = Path(ASSETS_DIR).expanduser().resolve()
    if REPO_ROOT is not None:
        assert not assets_root.is_relative_to(Path(REPO_ROOT).expanduser().resolve()), "Веса храните вне репозитория."
    assets_root.mkdir(parents=True, exist_ok=True)
    manifest_path = assets_root / "download_manifest.json"
    api = HfApi()
    if manifest_path.exists():
        prepared = json.loads(manifest_path.read_text(encoding="utf-8"))
        assert (prepared["model_id"], prepared["hmm_id"], prepared["dataset_id"], prepared["dataset_config"]) == (
            MODEL, HMM_CKPT, DATASET, "main"), "В этой папке уже готовятся другие артефакты; выберите новую папку."
        revisions = prepared["revisions"]
    else:
        assert not any((assets_root / name).exists() for name in ["model", "hmm", "gsm8k"]), (
            "В папке уже есть артефакты, но нет download_manifest.json; выберите новую папку, старые файлы не перезаписываются.")
        revisions = {"model": api.model_info(MODEL, revision=MODEL_REVISION).sha,
                     "hmm": api.model_info(HMM_CKPT, revision=HMM_REVISION).sha,
                     "dataset": api.dataset_info(DATASET, revision=DATASET_REVISION).sha}
        prepared = {"model_id": MODEL, "hmm_id": HMM_CKPT, "dataset_id": DATASET,
                    "dataset_config": "main", "revisions": revisions, "ready": False}

    def save_download_manifest(value):
        """value - сведения о фиксированных ревизиях/готовности; manifest_path - файл в ASSETS_DIR."""
        temporary = manifest_path.with_suffix(".json.tmp")
        with temporary.open("w", encoding="utf-8") as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, manifest_path)

    model_dir, hmm_dir, data_dir = assets_root / "model", assets_root / "hmm", assets_root / "gsm8k"
    model_info = api.model_info(MODEL, revision=revisions["model"], files_metadata=True)
    hmm_info = api.model_info(HMM_CKPT, revision=revisions["hmm"], files_metadata=True)
    model_files = list(model_info.siblings)
    hmm_files = [f for f in hmm_info.siblings if f.rfilename == "model.jpc"]
    assert len(hmm_files) == 1, "В HMM-репозитории не найден model.jpc."
    remaining = 0
    for directory, files in [(model_dir, model_files), (hmm_dir, hmm_files)]:
        for item in files:
            assert item.size is not None, f"Размер {item.rfilename} НЕ ПРОВЕРЕН; остановка до скачивания."
            target = directory / item.rfilename
            present = target.stat().st_size if target.is_file() else 0
            remaining += max(0, item.size - present)
    free = shutil.disk_usage(assets_root).free
    print(f"Осталось скачать модель/HMM: примерно {remaining / 1024**3:.2f} ГиБ; свободно {free / 1024**3:.2f} ГиБ.")
    assert free >= remaining + 1024**3, "Недостаточно места с запасом 1 ГиБ; выберите другой ASSETS_DIR."
    prepared["ready"] = False
    save_download_manifest(prepared)
    snapshot_download(repo_id=MODEL, revision=revisions["model"], local_dir=str(model_dir))
    hf_hub_download(repo_id=HMM_CKPT, filename="model.jpc", revision=revisions["hmm"], local_dir=str(hmm_dir))
    if data_dir.exists():
        local_data = datasets.load_from_disk(str(data_dir))
    else:
        local_data = datasets.load_dataset(DATASET, "main", revision=revisions["dataset"],
                                          cache_dir=str(assets_root / ".datasets_cache"))
        with tempfile.TemporaryDirectory(prefix=".gsm8k-", dir=assets_root) as temporary:
            staged = Path(temporary) / "dataset"
            local_data.save_to_disk(str(staged))
            os.replace(staged, data_dir)
    assert isinstance(local_data, datasets.DatasetDict) and {"train", "test"} <= set(local_data)
    assert len(local_data["test"]) >= N_TASKS
    assert (model_dir / "config.json").is_file() and (hmm_dir / "model.jpc").is_file()
    prepared.update({"ready": True, "split_sizes": {key: len(local_data[key]) for key in ["train", "test"]},
                     "preparation_packages": {key: importlib.metadata.version(key)
                                              for key in ["huggingface-hub", "datasets"]}})
    save_download_manifest(prepared)
    print("ГОТОВО. Перенесите model/, hmm/, gsm8k/ и download_manifest.json из:", assets_root)
    print("На DataSphere укажите эту папку как ASSETS_DIR и выключите RUN_LOCAL_DOWNLOAD.")

Осталось скачать модель/HMM: примерно 0.49 ГиБ; свободно 21.12 ГиБ.


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.jpc:   0%|          | 0.00/522M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7473 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1319 [00:00<?, ? examples/s]

ГОТОВО. Перенесите model/, hmm/, gsm8k/ и download_manifest.json из: C:\Users\MSI\Desktop\Other\yandex-education\yandex-studcamp\microdegree-math\project\CoDD-ablations\exp3-assets
На DataSphere укажите эту папку как ASSETS_DIR и выключите RUN_LOCAL_DOWNLOAD.


# 1. Окружение и установка

До открытия GPU-ядра подготовьте отдельное окружение по `environment.yml`, установите встроенный harness (`python -m pip install -e './lm-evaluation-harness[dev]'`) и `math_verify`. Не обновляйте torch/CUDA автоматически из этого блокнота. При несовместимости экспортированного CUDA-окружения с DataSphere подберите отдельное окружение и явно зафиксируйте отличие; его совместимость до теста - НЕ ПРОВЕРЕНО. Для метаданных и таблиц используются также pandas, IPython и huggingface-hub.

**Offline:** на машине с доступом к Hub сохраните полный snapshot `MODEL` (включая tokenizer/config и необходимые файлы custom code), HMM `model.jpc` из `HMM_CKPT`, а GSM8K `main` как DatasetDict через `save_to_disk`. Перенесите их вместе с checkout и установленными зависимостями в постоянное хранилище DataSphere. После ячейки 0.1 заполните `REPO_ROOT`, `STORAGE_ROOT` и `ASSETS_DIR`; три ручных локальных пути оставьте `None`, `OFFLINE=True`, `RUN_LOCAL_DOWNLOAD=False`. Ревизии прочитаются из `download_manifest.json`. Прежний режим с тремя ручными путями также сохранён: для него `ASSETS_DIR=None`. Нужны train и test: harness инициализирует описание few-shot даже при zero-shot. Учётные данные не записывать в блокнот.

**Online:** `OFFLINE=False`, локальные пути оставьте `None`, задайте cache при необходимости. Модель и HMM сначала разрешаются в immutable commit SHA, скачиваются один раз и далее загружаются только локально. Для GSM8K также фиксируется ревизия источника. Фиксированные модель и параметры эксперимента не меняются.

`check_environment()` не принимает аргументов: использует конфиг, проверяет CUDA/bfloat16, импорты и отпечатки исходников. `file_sha256(path)` хэширует один файл потоково. `canonical_bytes(value)` задаёт единую сериализацию для проверок целостности. `write_json_once(path, value)` создаёт JSON только при отсутствии или принимает уже совпадающий файл; при конфликте останавливается.

Отпечатки текстовых исходников нормализуют CRLF/LF, чтобы Windows checkout совпадал с Linux. Для референсного ipynb учитываются только ID, типы и исходники ячеек: выводы baseline не влияют на эту проверку. `notebook_source_sha256(path)` вычисляет такой отпечаток. Веса хэшируются как бинарные файлы без нормализации.


Для ZIP без `.git` используется `SOURCE_GIT_REVISION` из конфига как зафиксированная базовая ревизия; происхождение записывается в метаданные. Проверка содержимого исходников по SHA-256 сохраняется. Сам блокнот ищется по `NOTEBOOK_RELATIVE_PATH`. На DataSphere пропустите ячейку 0.1 и выполняйте начиная с 1.1 на новом ядре; пакеты для Linux/CUDA должны быть установлены отдельно.


In [ ]:
# 1.1. ОКРУЖЕНИЕ DATASPHERE - локально НЕ выполнять
import os
import sys
import json
import hashlib
import platform
import subprocess
import importlib.metadata
import random
import time
import copy
import tempfile
from contextlib import contextmanager
from dataclasses import dataclass
from unittest.mock import patch

if "ENVIRONMENT" not in globals() and ("torch" in sys.modules or "datasets" in sys.modules or "huggingface_hub" in sys.modules):
    raise RuntimeError("Начните с нового ядра: offline-настройки должны предшествовать импортам.")
assert REPO_ROOT is not None and STORAGE_ROOT is not None, "Заполните REPO_ROOT и STORAGE_ROOT."
REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
STORAGE_ROOT = Path(STORAGE_ROOT).expanduser().resolve()
assert REPO_ROOT.is_dir() and STORAGE_ROOT.is_dir(), "Пути должны существовать в хранилище проекта."
assert DEVICE.startswith("cuda") and PRECISION == "bfloat16"
assert (MODEL, HMM_CKPT, DATASET, SPLIT, N_TASKS) == (
    "GSAI-ML/LLaDA-8B-Instruct", "il18/llada-gsm-pc", "gsm8k", "test", 100)
assert (GEN_LENGTH, BLOCK_LENGTH, TEMPERATURE, NUM_STEPS_LIST) == (512, 32, 0.0, [64, 128])
assert REMASKING_LIST == ["low_confidence", "margin"] and SEEDS == [0, 1, 2]
assert (PC_TEMPERATURE, PC_FRAC, REVERSE_FRAC) == (0.2, 0.5, False)
assert (CAND_FROM, ORDER_FROM, CFG_SCALE, BATCH_SIZE, NUM_FEWSHOT) == ("pc", "base", 0.0, 1, 0)
os.environ["HF_HUB_OFFLINE"] = "1" if OFFLINE else "0"
os.environ["HF_DATASETS_OFFLINE"] = "1" if OFFLINE else "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
if HF_CACHE_DIR is not None:
    os.environ["HF_HOME"] = str(Path(HF_CACHE_DIR).expanduser().resolve())
sys.path.insert(0, str(REPO_ROOT / "lm-evaluation-harness"))
sys.path.insert(0, str(REPO_ROOT))


def canonical_bytes(value):
    """value - JSON-совместимый объект; возвращает однозначное UTF-8 представление."""
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"),
                      allow_nan=False).encode("utf-8")


def file_sha256(path):
    """path - существующий файл; читает блоками, не загружая веса целиком в RAM."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json_once(path, value):
    """path - целевой JSON; value - ожидаемые данные. Конфликт запрещает перезапись."""
    path = Path(path)
    if path.exists():
        assert json.loads(path.read_text(encoding="utf-8")) == value, f"Конфликт: {path}"
        return
    payload = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                     prefix=f".{path.name}.", delete=False) as stream:
        temporary = Path(stream.name)
        stream.write(payload)
        stream.flush()
        os.fsync(stream.fileno())
    try:
        # Один процесс: сверяем существующий файл; иначе публикуем целый JSON атомарным rename.
        if path.exists():
            assert json.loads(path.read_text(encoding="utf-8")) == value, f"Конфликт: {path}"
        else:
            os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


try:
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn.functional as F
    import pyjuice as juice
    import datasets
    from huggingface_hub import HfApi, snapshot_download, hf_hub_download
    from IPython.display import display, Markdown
    from eval.llada.codd_llada import CoddLlada
    from eval.llada.modeling_llada import LLaDAModelLM
    from eval.llada.llada_generate import (
        llada_diffusion_generate, get_num_transfer_tokens, add_gumbel_noise)
    from codd.model_class import CoDDOutput, apply_pc_logits
    from lm_eval.api.task import ConfigurableTask
    from lm_eval.api.instance import Instance
    from lm_eval.filters import build_filter_ensemble
    from lm_eval.models.huggingface import HFLM
    from lm_eval.models.utils import handle_stop_sequences, postprocess_generated_text
    from lm_eval.utils import load_yaml_config
except ImportError as exc:
    raise RuntimeError("Не хватает зависимостей. Подготовьте окружение по секции 1; "
                       "автоустановка отключена.") from exc


def notebook_source_sha256(path):
    """path - ipynb; учитывает IDs/типы/исходники ячеек, не outputs/execution_count."""
    notebook = json.loads(Path(path).read_text(encoding="utf-8"))
    sources = [{"id": cell["id"], "type": cell["cell_type"],
                "source": "".join(cell["source"]).replace("\r\n", "\n")}
               for cell in notebook["cells"]]
    return hashlib.sha256(canonical_bytes(sources)).hexdigest()


def check_environment():
    """Без аргументов: проверяет выбранную GPU, импорты checkout и совместимость исходников."""
    assert platform.system() == "Linux", "Запуск предусмотрен в Linux DataSphere."
    assert torch.cuda.is_available(), "CUDA недоступна - CPU fallback запрещён."
    torch.cuda.set_device(DEVICE)
    assert torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported(), (
        "Нужна аппаратная поддержка bfloat16 (NVIDIA Ampere или новее); эмуляция не принимается.")
    torch.set_grad_enabled(False)
    actual = {}
    for name, expected in EXPECTED_SOURCE_SHA256.items():
        source = REPO_ROOT / name
        assert source.is_file(), f"Нет исходника: {source}"
        actual[name] = (notebook_source_sha256(source) if source.suffix == ".ipynb" else
                        hashlib.sha256(source.read_bytes().replace(b"\r\n", b"\n")).hexdigest())
        assert actual[name] == expected, f"Изменился {name}: перепроверьте копии/API до запуска."
    for module in [sys.modules[CoddLlada.__module__], sys.modules[ConfigurableTask.__module__]]:
        assert Path(module.__file__).resolve().is_relative_to(REPO_ROOT), "Импортирован чужой checkout."
    versions = {}
    for name in ["torch", "transformers", "pyjuice", "datasets", "numpy", "pandas",
                 "huggingface-hub", "accelerate", "lm_eval"]:
        versions[name] = importlib.metadata.version(name)
    revision, revision_source = SOURCE_GIT_REVISION, "config: базовая ревизия ZIP, не подтверждена локальным Git"
    if (REPO_ROOT / ".git").exists():
        try:
            revision = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
                                      check=True, capture_output=True, text=True, timeout=15).stdout.strip()
            revision_source = "git"
        except (OSError, subprocess.SubprocessError):
            print("Git недоступен; используется SOURCE_GIT_REVISION и проверенные SHA-256 исходников.")
    assert isinstance(revision, str) and len(revision) == 40 and all(ch in "0123456789abcdef" for ch in revision), (
        "Для ZIP без Git заполните SOURCE_GIT_REVISION базовым commit SHA; неизвестное значение не подставляется.")
    return {"python": platform.python_version(), "packages": versions,
            "git_revision": revision, "git_revision_source": revision_source, "source_sha256": actual,
            "cuda_runtime": torch.version.cuda, "gpu": torch.cuda.get_device_name(),
            "gpu_capability": list(torch.cuda.get_device_capability()),
            "source_hash_definition": "SHA-256 LF-normalized source; notebooks: cell source/IDs only",
            "precision": PRECISION}


ENVIRONMENT = check_environment()
OUT_DIR = STORAGE_ROOT / "results" / "exp3_hmm_words_only"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_NOTEBOOK_SHA256 = EXPECTED_SOURCE_SHA256["notebooks/exp1_llada_default.ipynb"]
SMOKE_PASSED = False
PYJUICE_REPRODUCIBLE = None
SMOKE_REPORT = {"status": "НЕ ПРОВЕРЕНО", "reason": "Секция 5 ещё не выполнена."}
print(json.dumps(ENVIRONMENT, ensure_ascii=False, indent=2))
print("Убедитесь, что постоянное хранилище выбрано верно:", OUT_DIR)

# 2. Загрузка модели и HMM один раз

`WordsOnlyLlada.forward` сохраняет стандартные параметры оригинала: `input_ids` - текущие токены; `pc_block_range=(start,end)` - полуинтервал блока; `use_pc` - включение HMM; `cfg_scale` и `prompt_index` - classifier-free guidance и маска промпта; `pc_temperature` - температура оценок LLaDA перед HMM; `pc_frac` - порог доли масок; `reverse_frac` - направление сравнения; `**kwargs` - стандартные аргументы LLaDA. Новый `return_base_logits` включает выдачу обоих источников. При `False` вызывается оригинальный `CoddLlada.forward`.

В расширенном режиме трансформер вызывается один раз. **Клон создаётся до изменения на месте.** Поле `.logits` по-прежнему содержит результат CoDD; `.logits_base` - исходный клон, `.pc_active` - фактическое выполнение PC, `.mask_ratio` - доля масок до раскрытия. Логика `_pc_compiled` сохранена буквально: оригинал выставляет флаг даже если ранний вызов PC вернулся без активации. Здесь это не исправляется, чтобы не менять вторую ось абляции.

`_resolve_file(path_or_repo, filename)` в подклассе ищет только локальный файл; никаких скрытых запросов к Hub. `resolve_artifacts()` использует пути/режим из конфигурации и возвращает локальные артефакты с provenance. `artifact_manifest(directory)` хэширует все файлы snapshot, включая веса; это однократное дисковое чтение, оно может занять время. SHA-256 локальной HMM и датасета также сохраняются. Неизвестная локальная ревизия отмечается явно.

Модель загружается через `from_pretrained`: `torch_dtype=bfloat16` - точность весов, `device_map=DEVICE` - одна GPU, `attn_implementation="eager"` - как в оценке LLaDA форка, `local_files_only=True` - запрет сетевой загрузки после разрешения snapshot. HMM читается `juice.load(path)` и компилируется `juice.compile(pc_raw)` один раз. Генерация не использует `model.generate()` и его отдельный шаблон промпта.

In [ ]:
class WordsOnlyLlada(CoddLlada):
    @staticmethod
    def _resolve_file(path_or_repo, filename):
        """path_or_repo - локальный snapshot; filename - имя файла, сеть не используется."""
        candidate = Path(path_or_repo) / filename
        return str(candidate) if candidate.is_file() else None

    def forward(self, input_ids=None, pc_block_range=None, use_pc=True,
                cfg_scale=0.0, prompt_index=None, pc_temperature=None,
                pc_frac=None, reverse_frac=None, return_base_logits=False, **kwargs):
        """Параметры оригинального forward описаны выше; return_base_logits добавляет диагностику."""
        if not return_base_logits:
            return super().forward(
                input_ids=input_ids, pc_block_range=pc_block_range, use_pc=use_pc,
                cfg_scale=cfg_scale, prompt_index=prompt_index,
                pc_temperature=pc_temperature, pc_frac=pc_frac,
                reverse_frac=reverse_frac, **kwargs)
        cfg = self.codd_config
        if cfg_scale > 0.0 and prompt_index is not None:
            un_x = input_ids.clone()
            un_x[prompt_index] = cfg.mask_id
            output = LLaDAModelLM.forward(self, torch.cat([input_ids, un_x], dim=0), **kwargs)
            logits, un_logits = torch.chunk(output.logits, 2, dim=0)
            logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
        else:
            output = LLaDAModelLM.forward(self, input_ids, **kwargs)
            logits = output.logits
        logits_base = logits.clone()
        pc_active = False
        mask_ratio = None
        if use_pc and self.pc_model is not None and pc_block_range is not None:
            pct = pc_temperature if pc_temperature is not None else cfg.pc_temperature
            pcf = pc_frac if pc_frac is not None else cfg.pc_frac
            rf = reverse_frac if reverse_frac is not None else cfg.reverse_frac
            start, end = pc_block_range
            ratio = (input_ids[:, start:end] == cfg.mask_id).float().mean()
            mask_ratio = float(ratio.item())
            pc_active = bool((not rf and ratio < pcf) or (rf and ratio > pcf))
            record = not self._pc_compiled
            logits = apply_pc_logits(
                pc_model=self.pc_model, input_ids=input_ids, logits=logits,
                start_idx=start, end_idx=end, mask_id=cfg.mask_id,
                vocab_size=cfg.vocab_size, pc_temperature=pct, pc_frac=pcf,
                reverse_frac=rf, record_cudagraph=record)
            if record:
                self._pc_compiled = True
        result = CoDDOutput(logits=logits)
        result.logits_base = logits_base
        result.pc_active = pc_active
        result.mask_ratio = mask_ratio
        return result


def artifact_manifest(directory):
    """directory - локальный snapshot/датасет. Возвращает SHA-256 и размеры всех его файлов."""
    directory = Path(directory)
    manifest = {}
    for path in sorted(directory.rglob("*")):
        relative = path.relative_to(directory)
        if path.is_file() and not any(part.startswith(".") for part in relative.parts):
            manifest[str(relative)] = {"sha256": file_sha256(path), "bytes": path.stat().st_size}
    assert manifest, f"Пустой каталог артефакта: {directory}"
    return manifest


def resolve_artifacts():
    """Без аргументов: offline использует указанные файлы, online сначала фиксирует ревизии Hub."""
    if OFFLINE:
        if ASSETS_DIR is not None:
            assert all(v is None for v in [MODEL_LOCAL_DIR, HMM_LOCAL_PATH, DATASET_LOCAL_DIR]), (
                "Выберите ASSETS_DIR либо три ручных пути, не смешивайте способы.")
            assets_root = Path(ASSETS_DIR).expanduser().resolve()
            prepared = json.loads((assets_root / "download_manifest.json").read_text(encoding="utf-8"))
            assert prepared["ready"] is True, "Локальная подготовка не завершена: повторите ячейку 0.1."
            assert (prepared["model_id"], prepared["hmm_id"], prepared["dataset_id"], prepared["dataset_config"]) == (
                MODEL, HMM_CKPT, DATASET, "main"), "Перенесены артефакты другого эксперимента."
            model_dir, hmm_path, data_dir = assets_root / "model", assets_root / "hmm/model.jpc", assets_root / "gsm8k"
            revisions = prepared["revisions"]
        else:
            assert all(v is not None for v in [MODEL_LOCAL_DIR, HMM_LOCAL_PATH, DATASET_LOCAL_DIR]), (
                "Offline: заполните ASSETS_DIR либо MODEL_LOCAL_DIR, HMM_LOCAL_PATH, DATASET_LOCAL_DIR.")
            model_dir = Path(MODEL_LOCAL_DIR).expanduser().resolve()
            hmm_path = Path(HMM_LOCAL_PATH).expanduser().resolve()
            data_dir = Path(DATASET_LOCAL_DIR).expanduser().resolve()
            revisions = {"model": LOCAL_MODEL_REVISION, "hmm": LOCAL_HMM_REVISION,
                         "dataset": LOCAL_DATASET_REVISION}
            revisions = {k: v or "НЕ ПРОВЕРЕНО: локальная ревизия неизвестна; см. SHA-256"
                         for k, v in revisions.items()}
        assert model_dir.is_dir() and (model_dir / "config.json").is_file()
        assert hmm_path.is_file() and data_dir.is_dir()
    else:
        assert all(v is None for v in [MODEL_LOCAL_DIR, HMM_LOCAL_PATH, DATASET_LOCAL_DIR]), (
            "Online-режим не смешивается с локальными путями; выберите один способ загрузки.")
        api = HfApi()
        revisions = {"model": api.model_info(MODEL, revision=MODEL_REVISION).sha,
                     "hmm": api.model_info(HMM_CKPT, revision=HMM_REVISION).sha,
                     "dataset": api.dataset_info(DATASET, revision=DATASET_REVISION).sha}
        model_dir = Path(snapshot_download(repo_id=MODEL, revision=revisions["model"]))
        hmm_path = Path(hf_hub_download(repo_id=HMM_CKPT, filename="model.jpc",
                                      revision=revisions["hmm"]))
        data_dir = None
    # Не допускаем загрузку/компиляцию случайной встроенной PC при from_pretrained.
    assert not (model_dir / "pc_model.jpc").exists(), "Нужен snapshot базовой LLaDA, без встроенной PC."
    assert not (model_dir / "codd_config.json").exists(), "Нужен snapshot базовой LLaDA, без CoDD config."
    provenance = {"model_id": MODEL, "hmm_id": HMM_CKPT, "revisions": revisions,
                  "model_files": artifact_manifest(model_dir),
                  "hmm_file": {"sha256": file_sha256(hmm_path), "bytes": hmm_path.stat().st_size}}
    if data_dir is not None:
        provenance["dataset_files"] = artifact_manifest(data_dir)
    return model_dir, hmm_path, data_dir, provenance


assert "model" not in globals(), "Модель уже загружена. Для повторной загрузки перезапустите ядро."
MODEL_DIR, HMM_PATH, DATA_DIR, ARTIFACTS = resolve_artifacts()
model = WordsOnlyLlada.from_pretrained(
    str(MODEL_DIR), torch_dtype=torch.bfloat16, device_map=DEVICE,
    attn_implementation="eager", local_files_only=True)
assert model.pc_model is None
pc_raw = juice.load(str(HMM_PATH))
model.pc_model = juice.compile(pc_raw).to(DEVICE)
del pc_raw
model.eval()
model.requires_grad_(False)
assert model.device.type == "cuda"
assert next(model.parameters()).dtype == torch.bfloat16
assert model.pc_model is not None
assert model.tokenizer is not None and model.codd_config.mask_id == 126336
assert model.get_input_embeddings().weight.shape[0] > model.codd_config.mask_id
model.codd_config.pc_temperature = PC_TEMPERATURE
model.codd_config.pc_frac = PC_FRAC
model.codd_config.reverse_frac = REVERSE_FRAC
model.codd_config.block_length = BLOCK_LENGTH
model.codd_config.gen_length = GEN_LENGTH
model.codd_config.temperature = TEMPERATURE
model.codd_config.cfg_scale = CFG_SCALE
ARTIFACTS["runtime_dtypes"] = {"model": str(next(model.parameters()).dtype),
                               "pc_input_params": str(model.pc_model.input_layer_group[0].params.dtype),
                               "external_evidence": "torch.float32",
                               "low_confidence_softmax": "torch.float64"}
print("Модель и HMM загружены один раз. GPU-эквивалентность ещё НЕ ПРОВЕРЕНА.")

# 3. Данные и задачи с хэшем - протокол exp1

`load_gsm8k()` читает `DATA_DIR` через `load_from_disk` в offline или `gsm8k/main` с зафиксированной ревизией в online. Конфигурация задачи, шаблон вопроса, фильтры и exact_match импортируются из YAML/harness, не переписываются. `ConfigurableTask(config=task_config)` использует эту конфигурацию; `custom_dataset` - только замена источника данных, без изменения порядка строк.

Как в exp1, `task.build_all_requests` получает `limit=N_TASKS` (первые 100), `rank=0/world_size=1` (один процесс), выключенный cache запросов, `system_instruction` (системный текст), `apply_chat_template=True` (шаблон модели), `fewshot_as_multiturn=False` (zero-shot) и `chat_template=lm.apply_chat_template` (тот же fallback системной роли, что у HFLM). `lm` - HFLM над уже загруженной моделью, он не загружает вторые веса и не выполняет генерацию.

`lm.tok_batch_encode` кодирует один промпт без усечения; если промпт с ответом не помещается, выполнение останавливается. `UNTIL` берётся из задачи с добавлением EOS через `handle_stop_sequences(until, eos)` - готовую функцию harness. В tasks.json сохраняются поля exp1 (`ids_sha256`, `questions_sha256`, `tasks[*].prompt_sha256`) и дополнительные документы, тексты промптов и token IDs.

Если рядом есть результаты exp1, сравниваются только конфиг, вопросы и промпты. Ответы и точность другого эксперимента не читаются и не сравниваются. При несовпадении протокола - остановка; при отсутствии файлов сопоставимость помечается НЕ ПРОВЕРЕНО.

In [ ]:
def load_gsm8k():
    """Без аргументов: возвращает DatasetDict train/test из настроенного источника."""
    if OFFLINE:
        loaded = datasets.load_from_disk(str(DATA_DIR))
    else:
        loaded = datasets.load_dataset(DATASET, "main", revision=ARTIFACTS["revisions"]["dataset"])
    assert isinstance(loaded, datasets.DatasetDict), "Нужен DatasetDict с train и test."
    assert {"train", SPLIT} <= set(loaded), "Нужны train и test, как в референсе exp1."
    for split in ["train", SPLIT]:
        assert {"question", "answer"} <= set(loaded[split].column_names)
    assert len(loaded[SPLIT]) >= N_TASKS
    return loaded


gsm8k_data = load_gsm8k()
TASK_CONFIG = load_yaml_config(str(REPO_ROOT / "lm-evaluation-harness/lm_eval/tasks/gsm8k/gsm8k.yaml"))
task_config = copy.deepcopy(TASK_CONFIG)
task_config["custom_dataset"] = lambda **kwargs: gsm8k_data
task_config["dataset_kwargs"] = {}
task = ConfigurableTask(config=task_config)
task.set_config(key="num_fewshot", value=NUM_FEWSHOT)
task.set_fewshot_seed(0)
assert task.OUTPUT_TYPE == "generate_until"
assert {f.name for f in task._filters} == {"strict-match", "flexible-extract"}
assert task.get_config("generation_kwargs").get("do_sample") is False
assert float(task.get_config("generation_kwargs").get("temperature", 0.0)) == TEMPERATURE
lm = HFLM(pretrained=model, tokenizer=model.tokenizer, batch_size=BATCH_SIZE)
assert lm.backend == "causal"
task.build_all_requests(
    limit=N_TASKS, rank=0, world_size=1, cache_requests=False,
    rewrite_requests_cache=False, system_instruction=SYSTEM_INSTRUCTION,
    apply_chat_template=True, fewshot_as_multiturn=False,
    chat_template=lm.apply_chat_template, tokenizer_name=lm.tokenizer_name)
INSTANCES = list(task.instances)
TASK_IDS = [inst.doc_id for inst in INSTANCES]
assert TASK_IDS == list(range(N_TASKS))
assert all(inst.request_type == "generate_until" and inst.repeats == 1 for inst in INSTANCES)
assert all(inst.args[1] == INSTANCES[0].args[1] for inst in INSTANCES)
UNTIL = handle_stop_sequences(
    copy.deepcopy(INSTANCES[0].args[1]).pop("until", None),
    eos=lm.tok_decode(lm.eot_token_id, skip_special_tokens=False))
PROMPT_IDS = []
for inst in INSTANCES:
    ids, _ = lm.tok_batch_encode([inst.args[0]], left_truncate_len=None, truncation=False)
    assert ids.shape[0] == 1 and model.codd_config.mask_id not in ids[0].tolist()
    PROMPT_IDS.append(ids[0].tolist())
limits = [lm.max_length]
if getattr(model.config, "max_sequence_length", None):
    limits.append(int(model.config.max_sequence_length))
assert max(map(len, PROMPT_IDS)) + GEN_LENGTH <= min(limits), "Промпт с ответом не помещается; усечение запрещено."
IDS_SHA256 = hashlib.sha256(json.dumps(TASK_IDS, separators=(",", ":")).encode("utf-8")).hexdigest()
QUESTIONS_SHA256 = hashlib.sha256(json.dumps(
    [inst.doc["question"] for inst in INSTANCES], ensure_ascii=False,
    separators=(",", ":")).encode("utf-8")).hexdigest()
TASKS_DOC = {
    "dataset": DATASET, "split": SPLIT, "n_tasks": N_TASKS,
    "system_instruction": SYSTEM_INSTRUCTION, "num_fewshot": NUM_FEWSHOT,
    "chat_template_applied": True, "ids": TASK_IDS, "ids_sha256": IDS_SHA256,
    "ids_sha256_definition": 'sha256(json.dumps(ids, separators=(",",":")).encode("utf-8"))',
    "questions_sha256": QUESTIONS_SHA256,
    "documents_sha256": hashlib.sha256(canonical_bytes([inst.doc for inst in INSTANCES])).hexdigest(),
    "tasks": [{"task_id": inst.doc_id, "question": inst.doc["question"],
               "answer": inst.doc["answer"], "document": inst.doc,
               "prompt": inst.args[0], "prompt_ids": PROMPT_IDS[inst.doc_id],
               "prompt_sha256": hashlib.sha256(inst.args[0].encode("utf-8")).hexdigest()}
              for inst in INSTANCES]}
write_json_once(OUT_DIR / "tasks.json", TASKS_DOC)
ARTIFACTS["dataset_fingerprint"] = getattr(gsm8k_data[SPLIT], "_fingerprint", "НЕ ПРОВЕРЕНО")
COMMON_PARAMS = {
    "MODEL": MODEL, "HMM_CKPT": HMM_CKPT, "DATASET": DATASET, "DATASET_SPLIT": SPLIT,
    "N_TASKS": N_TASKS, "GEN_LENGTH": GEN_LENGTH, "BLOCK_LENGTH": BLOCK_LENGTH,
    "TEMPERATURE": TEMPERATURE, "NUM_STEPS_LIST": NUM_STEPS_LIST,
    "REMASKING_LIST": REMASKING_LIST, "PC_TEMPERATURE": PC_TEMPERATURE,
    "PC_FRAC": PC_FRAC, "REVERSE_FRAC": REVERSE_FRAC, "SEEDS": SEEDS}
COMMON_SHA256 = hashlib.sha256(canonical_bytes(COMMON_PARAMS)).hexdigest()
BASELINE_PROTOCOL_CHECK = {
    "status": "НЕ ПРОВЕРЕНО", "tasks": "Нет tasks.json exp1 в хранилище.",
    "parameters": "Нет config.json exp1 в хранилище."}
baseline_dir = Path(BASELINE_RESULTS_DIR) if BASELINE_RESULTS_DIR is not None else STORAGE_ROOT / "results/exp1_llada_default"
if (baseline_dir / "tasks.json").is_file():
    reference_tasks = json.loads((baseline_dir / "tasks.json").read_text(encoding="utf-8"))
    for key in ["dataset", "split", "n_tasks", "ids_sha256", "questions_sha256",
                "system_instruction", "num_fewshot", "chat_template_applied"]:
        assert reference_tasks[key] == TASKS_DOC[key], f"Протокол exp1 отличается: {key}"
    assert {t["task_id"]: t["prompt_sha256"] for t in reference_tasks["tasks"]} == {
        t["task_id"]: t["prompt_sha256"] for t in TASKS_DOC["tasks"]}, "Отличаются промпты exp1."
    BASELINE_PROTOCOL_CHECK["tasks"] = "MATCH"
if (baseline_dir / "config.json").is_file():
    reference_config = json.loads((baseline_dir / "config.json").read_text(encoding="utf-8"))
    assert reference_config["common_params"] == COMMON_PARAMS, "Отличаются общие параметры exp1."
    for key, value in {"DTYPE": PRECISION, "ATTN_IMPLEMENTATION": "eager",
                       "NUM_FEWSHOT": NUM_FEWSHOT, "SYSTEM_INSTRUCTION": SYSTEM_INSTRUCTION}.items():
        assert reference_config["exp_params"][key] == value, f"Настройка exp1 отличается: {key}"
    BASELINE_PROTOCOL_CHECK["parameters"] = "MATCH"
if BASELINE_PROTOCOL_CHECK["tasks"] == BASELINE_PROTOCOL_CHECK["parameters"] == "MATCH":
    BASELINE_PROTOCOL_CHECK["status"] = "MATCH"
elif "MATCH" in [BASELINE_PROTOCOL_CHECK["tasks"], BASELINE_PROTOCOL_CHECK["parameters"]]:
    BASELINE_PROTOCOL_CHECK["status"] = "PARTIAL: часть протокола НЕ ПРОВЕРЕНА"
print("sha256 списка id:", IDS_SHA256)
print("sha256 вопросов:", QUESTIONS_SHA256)
print("Сверка протокола exp1:", BASELINE_PROTOCOL_CHECK)
print("Промпт 0:\n", INSTANCES[0].args[0])

# 4. Функции: hmm-words-only, оценка, журнал и возобновление

`llada_words_only_generate` - локальная копия цикла `llada_diffusion_generate`. Параметры:
- `model`, `prompt` - загруженная модель и tensor токенов промпта `(1,L)` на GPU;
- `num_steps` - общее число шагов по всем блокам; `gen_length` - новые токены; `block_length` - длина блока; `mask_id` - ID маски;
- `temperature` - Gumbel-температура, здесь строго 0; `cfg_scale` - CFG, здесь строго 0;
- `remasking` - формула уверенности; `use_pc` - включение HMM; `pc_temperature`, `pc_frac`, `reverse_frac` - параметры PC, описанные в разделе 2;
- `cand_from`, `order_from` - источники токенов и уверенности; `collect_trace` - собирать диагностический журнал без дополнительных вызовов модели/HMM.

`confidence_scores(logits, candidates, remasking, shared_random)` получает выбранные token IDs и источник оценок. `shared_random` нужен только для `random`: один и тот же вектор для двух ранжирований. `select_positions(scores, mask_index, end_idx, count)` повторяет исходные ограничения и top-k; `count` - число раскрываемых токенов на шаге, `end_idx` - конец доступного блока. `order_distance(left, right)` считает расстояние Жаккара наборов позиций, для двух пустых - 0.

`GenerationResult` содержит итоговые `tokens`, список `trace`, число `pc_active_steps` и выполненных `steps`. Записи журнала имеют поля exp1 `block`, `step_in_block`, `global_step`, `n_masked_block_before`, `positions_abs`, `positions_rel`, `confidence`, `token_ids`, а также абсолютные/относительные позиции, источники, маски и две delta-метрики. Диагностика касается только текущего состояния, а не воображаемого независимого прогона LLaDA.

In [ ]:
@dataclass
class GenerationResult:
    tokens: torch.Tensor
    trace: list
    pc_active_steps: int
    steps: int


def confidence_scores(logits, candidates, remasking, shared_random=None):
    """logits - источник оценок; candidates - token IDs; remasking - правило; shared_random - общий U(0,1)."""
    if remasking == "low_confidence":
        probs = F.softmax(logits.to(torch.float64), dim=-1)
        return torch.squeeze(torch.gather(probs, dim=-1, index=candidates.unsqueeze(-1)), -1)
    if remasking == "random":
        assert shared_random is not None and shared_random.shape == candidates.shape
        return shared_random
    if remasking == "margin":
        probs = torch.softmax(logits, dim=-1)
        top_probs, _ = torch.topk(probs, k=2, dim=-1)
        return top_probs[..., 0] - top_probs[..., 1]
    if remasking == "entropy":
        probs = torch.softmax(logits, dim=-1)
        return torch.sum(probs * torch.log(probs + 1e-10), dim=-1)
    raise NotImplementedError(remasking)


def select_positions(scores, mask_index, end_idx, count):
    """scores/mask_index - (1,L); end_idx - конец блока; count - k исходного top-k."""
    assert scores.shape[0] == mask_index.shape[0] == 1
    scores = scores.clone()
    scores[:, end_idx:] = -np.inf
    confidence = torch.where(mask_index, scores, -np.inf)
    if count == 0:
        return torch.empty(0, device=scores.device, dtype=torch.long)
    assert count > 0 and int(torch.isfinite(confidence).sum()) >= count, "Недостаточно допустимых позиций."
    return torch.topk(confidence[0], k=count).indices


def order_distance(left, right):
    """left/right - наборы token-позиций (tensor или iterable); возвращает расстояние Жаккара."""
    left, right = set(map(int, left)), set(map(int, right))
    return 0.0 if not (left | right) else 1.0 - len(left & right) / len(left | right)


@torch.no_grad()
def llada_words_only_generate(
    model, prompt, num_steps=128, gen_length=512, block_length=32,
    temperature=0.0, cfg_scale=0.0, remasking="low_confidence", mask_id=126336,
    use_pc=True, pc_temperature=0.2, pc_frac=0.5, reverse_frac=False,
    cand_from="pc", order_from="base", collect_trace=True,
):
    """Значения всех параметров описаны в разделе 4; returns GenerationResult без повторного forward для логов."""
    assert cand_from in {"base", "pc"} and order_from in {"base", "pc"}
    assert temperature == 0.0 and cfg_scale == 0.0, "Этот эксперимент проверяет только temperature=cfg_scale=0."
    assert prompt.ndim == 2 and prompt.shape[0] == 1 and prompt.device.type == "cuda"
    assert gen_length > 0 and block_length > 0 and num_steps > 0
    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    assert num_steps % num_blocks == 0
    steps_per_block = num_steps // num_blocks
    assert steps_per_block <= block_length, "В этом протоколе нет шагов без раскрытия."
    assert not model.training and model.device.type == "cuda"
    assert not use_pc or model.pc_model is not None, "Запрошена HMM, но она отсутствует."
    x = torch.full((1, prompt.shape[1] + gen_length), mask_id, dtype=torch.long, device=model.device)
    x[:, :prompt.shape[1]] = prompt.clone()
    prompt_index = (x != mask_id)
    trace, active_steps, global_step = [], 0, 0
    for block in range(num_blocks):
        start_idx = prompt.shape[1] + block * block_length
        end_idx = start_idx + block_length
        block_mask_index = (x[:, start_idx:end_idx] == mask_id)
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps_per_block)
        for step in range(steps_per_block):
            mask_index = (x == mask_id)
            before = mask_index[:, start_idx:end_idx].clone()
            ratio = float(before.float().mean().item())
            output = model(
                x, pc_block_range=(start_idx, end_idx) if use_pc else None,
                use_pc=use_pc, cfg_scale=cfg_scale, prompt_index=None,
                pc_temperature=pc_temperature, pc_frac=pc_frac,
                reverse_frac=reverse_frac, return_base_logits=True)
            logits_base, logits_pc = output.logits_base, output.logits
            expected_active = use_pc and ((not reverse_frac and ratio < pc_frac)
                                          or (reverse_frac and ratio > pc_frac))
            assert output.pc_active == expected_active
            if not output.pc_active:
                assert torch.equal(logits_base, logits_pc), "Неактивная HMM изменила логиты."
            sources = {"base": logits_base, "pc": logits_pc}
            candidates = torch.argmax(add_gumbel_noise(sources[cand_from], temperature), dim=-1)
            shared_random = torch.rand(candidates.shape, device=candidates.device) if remasking == "random" else None
            scores_base = confidence_scores(logits_base, candidates, remasking, shared_random)
            scores_pc = confidence_scores(logits_pc, candidates, remasking, shared_random)
            count = int(num_transfer_tokens[0, step].item())
            selected_base = select_positions(scores_base, mask_index, end_idx, count)
            selected_pc = select_positions(scores_pc, mask_index, end_idx, count)
            selected = selected_base if order_from == "base" else selected_pc
            actual_scores = scores_base if order_from == "base" else scores_pc
            assert bool(((selected >= start_idx) & (selected < end_idx)).all())
            x0 = torch.where(mask_index, candidates, x)
            tokens = x0[0, selected]
            assert not bool((tokens == mask_id).any()), "Кандидат равен MASK; расписание перестанет соответствовать оригиналу."
            delta_order = order_distance(selected_base.tolist(), selected_pc.tolist())
            if collect_trace:
                arg_base = torch.argmax(logits_base[:, start_idx:end_idx], dim=-1)
                arg_pc = torch.argmax(logits_pc[:, start_idx:end_idx], dim=-1)
                denominator = int(before.sum().item())
                delta_words = (float(((arg_base != arg_pc) & before).sum().item()) / denominator
                               if denominator else 0.0)
                if not output.pc_active:
                    assert delta_words == 0.0 and delta_order == 0.0
                trace.append({
                    "block": block, "step": step, "step_in_block": step, "global_step": global_step,
                    "start_idx": start_idx, "end_idx": end_idx,
                    "positions": selected.tolist(), "positions_abs": selected.tolist(),
                    "positions_rel": (selected - prompt.shape[1]).tolist(),
                    "positions_in_block": (selected - start_idx).tolist(),
                    "confidence": actual_scores[0, selected].tolist(), "token_ids": tokens.tolist(),
                    "selected_base": selected_base.tolist(), "selected_pc": selected_pc.tolist(),
                    "mask_ratio": ratio, "masked_before": denominator, "n_masked_block_before": denominator,
                    "pc_active": output.pc_active, "delta_words": delta_words,
                    "delta_order": delta_order, "cand_from": cand_from, "order_from": order_from})
            x[0, selected] = tokens
            active_steps += int(output.pc_active)
            global_step += 1
            del output, sources, logits_base, logits_pc, scores_base, scores_pc, actual_scores
        assert not bool((x[:, start_idx:end_idx] == mask_id).any()), "Блок не раскрыт полностью."
    assert torch.equal(x[:, :prompt.shape[1]], prompt)
    assert not bool((x[:, prompt.shape[1]:] == mask_id).any())
    assert global_step == num_steps
    return GenerationResult(x, trace, active_steps, global_step)

## 4.1. Оценка ответа и seed

`task_seed(seed, task_id)` детерминированно получает 32-битный seed из SHA-256 строки `exp3:{seed}:{task_id}`. Источники remasking/числа шагов в derivation не входят. `seed_all(value)` устанавливает Python/NumPy/PyTorch/CUDA seed. Это позволяет не проигрывать предыдущие задачи после обрыва; возможность воспроизвести сами сэмплы PyJuice проверяется отдельно.

`score_response(inst, text)` применяет готовые `task._filters` к одному запросу, затем `task.process_results` к его документу; `inst` - объект запроса harness, `text` - декодированный ответ. Оба фильтра выполняются, основной `correct` берётся из `flexible-extract`. Собственного regex или числовой нормализации нет.

`process_task(task_id, remasking, num_steps, seed, run_sha256)` выполняет одну задачу: ID/правило/шаги/экспериментальный seed определяют запуск, `run_sha256` связывает запись с метаданными. Синхронизация GPU до и после генерации делает `sec` измерением генерации с диагностикой; время загрузки, оценки и записи файлов туда не входит.

**Различие схем с exp1:** по утверждённому плану exp3 `prediction` хранит полный декодированный ответ; `extracted_answer` - результат flexible-extract. В exp1 `prediction` содержит уже извлечённый ответ. Для общих потребителей также есть `generation`, `correct_strict`, `prediction_strict`, `sec`, `gen_length`, `prompt_tokens`. Это отличие записано в метаданные; автоматически смешивать эти поля нельзя.

In [ ]:
def task_seed(seed, task_id):
    """seed - номер повтора; task_id - индекс test; результат не зависит от remasking/steps."""
    digest = hashlib.sha256(f"exp3:{seed}:{task_id}".encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "little")


def seed_all(value):
    """value - 32-битный seed; PyJuice может использовать неподконтрольную случайность, см. секцию 5."""
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)
    torch.cuda.manual_seed_all(value)


def score_response(inst, text):
    """inst - запрос harness; text - ответ; возвращает словари exact_match и строк по двум фильтрам."""
    probe = copy.copy(inst)
    probe.resps = [text]
    probe.filtered_resps = {}
    for ensemble in task._filters:
        ensemble.apply([probe])
    scores, predictions = {}, {}
    for name in ["strict-match", "flexible-extract"]:
        extracted = probe.filtered_resps[name]
        assert isinstance(extracted, str)
        metrics = task.process_results(probe.doc, [extracted])
        value = float(metrics["exact_match"])
        assert value in (0.0, 1.0)
        scores[name], predictions[name] = int(value), extracted
    return scores, predictions


def validate_trace(trace, generated_ids, prompt_length, num_steps, gen_length,
                   remasking, use_pc=True):
    """trace - шаги; generated_ids - ответ; остальные параметры задают ожидаемое расписание, use_pc - режим HMM."""
    assert len(trace) == num_steps and len(generated_ids) == gen_length
    num_blocks = gen_length // BLOCK_LENGTH
    per_block = num_steps // num_blocks
    assert num_steps % num_blocks == 0 and per_block <= BLOCK_LENGTH
    reconstructed = [None] * gen_length
    for index, rec in enumerate(trace):
        block, step = divmod(index, per_block)
        start = prompt_length + block * BLOCK_LENGTH
        end = start + BLOCK_LENGTH
        remaining = sum(v is None for v in reconstructed[block * BLOCK_LENGTH:(block + 1) * BLOCK_LENGTH])
        count = BLOCK_LENGTH // per_block + int(step < BLOCK_LENGTH % per_block)
        assert (rec["global_step"], rec["block"], rec["step_in_block"]) == (index, block, step)
        assert (rec["start_idx"], rec["end_idx"], rec["n_masked_block_before"]) == (start, end, remaining)
        assert rec["mask_ratio"] == remaining / BLOCK_LENGTH
        expected_active = use_pc and ((not REVERSE_FRAC and remaining / BLOCK_LENGTH < PC_FRAC)
                                      or (REVERSE_FRAC and remaining / BLOCK_LENGTH > PC_FRAC))
        assert rec["pc_active"] == expected_active
        positions = rec["positions_abs"]
        assert len(positions) == len(set(positions)) == len(rec["token_ids"]) == len(rec["confidence"]) == count
        assert rec["positions_rel"] == [pos - prompt_length for pos in positions]
        assert rec["positions_in_block"] == [pos - start for pos in positions]
        eligible = {prompt_length + j for j, token in enumerate(reconstructed)
                    if token is None and start <= prompt_length + j < end}
        for key in ["selected_base", "selected_pc"]:
            assert len(rec[key]) == len(set(rec[key])) == count
            assert set(rec[key]) <= eligible
        assert positions == rec["selected_base" if rec["order_from"] == "base" else "selected_pc"]
        assert rec["delta_order"] == order_distance(rec["selected_base"], rec["selected_pc"])
        assert 0.0 <= rec["delta_words"] <= 1.0 and 0.0 <= rec["delta_order"] <= 1.0
        if not expected_active:
            assert rec["delta_words"] == rec["delta_order"] == 0.0
        for pos, token, confidence in zip(positions, rec["token_ids"], rec["confidence"]):
            assert pos in eligible and np.isfinite(confidence)
            assert isinstance(token, int) and 0 <= token < model.codd_config.vocab_size
            assert token != model.codd_config.mask_id
            reconstructed[pos - prompt_length] = token
    assert reconstructed == generated_ids, "Токены журнала не восстанавливают ответ."


def process_task(task_id, remasking, num_steps, seed, run_sha256):
    """task_id/seed - задача и повтор; remasking/num_steps - конфигурация; run_sha256 - идентификатор запуска."""
    inst = INSTANCES[task_id]
    derived = task_seed(seed, task_id)
    seed_all(derived)
    prompt = torch.tensor([PROMPT_IDS[task_id]], device=model.device, dtype=torch.long)
    torch.cuda.synchronize()
    started = time.perf_counter()
    result = llada_words_only_generate(
        model, prompt, num_steps=num_steps, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
        temperature=TEMPERATURE, cfg_scale=CFG_SCALE, remasking=remasking,
        mask_id=model.codd_config.mask_id, use_pc=True,
        pc_temperature=PC_TEMPERATURE, pc_frac=PC_FRAC, reverse_frac=REVERSE_FRAC,
        cand_from=CAND_FROM, order_from=ORDER_FROM, collect_trace=True)
    torch.cuda.synchronize()
    seconds = time.perf_counter() - started
    ids = result.tokens[0, prompt.shape[1]:].tolist()
    validate_trace(result.trace, ids, prompt.shape[1], num_steps, GEN_LENGTH, remasking)
    text = postprocess_generated_text(lm.tok_decode(ids), stop=UNTIL, think_end_token=None)
    scores, predictions = score_response(inst, text)
    row = {"task_id": task_id, "seed": seed, "task_seed": derived,
           "correct": scores["flexible-extract"], "prediction": text,
           "extracted_answer": predictions["flexible-extract"],
           "generation": text, "correct_strict": scores["strict-match"],
           "prediction_strict": predictions["strict-match"],
           "num_steps": num_steps, "remasking": remasking,
           "pc_active_steps": result.pc_active_steps,
           "delta_words": float(np.mean([s["delta_words"] for s in result.trace])),
           "delta_order": float(np.mean([s["delta_order"] for s in result.trace])),
           "gen_length": GEN_LENGTH, "prompt_tokens": prompt.shape[1],
           "generation_ids": ids, "sec": seconds, "run_sha256": run_sha256}
    step_rows, cumulative = [], 0
    for rec in result.trace:
        cumulative += int(rec["pc_active"])
        step_rows.append({"task_id": task_id, "seed": seed, "task_seed": derived,
                          "remasking": remasking, "num_steps": num_steps,
                          "run_sha256": run_sha256, "pc_active_steps": cumulative, **rec})
    return row, step_rows

## 4.2. Надёжная запись и восстановление

`combo_paths(directory, remasking, num_steps, seed)` задаёт два JSONL для одной конфигурации. Имена: `{remasking}_steps{S}_seed{SEED}.jsonl` и `steplog_{remasking}_steps{S}_seed{SEED}.jsonl`.

`commit_task(directory, row, step_rows, fail_after=None)` сначала дописывает все шаги задачи и делает fsync, потом результат с контрольной суммой **точных байтов** журнала и снова fsync. Только строка результата означает завершение. `fail_after` - исключительно тестовый разрыв после `trace` или в середине `result_bytes`; основной прогон его не задаёт.

`read_jsonl_prefix(path)` читает полные строки, возвращая offsets/сырые байты; повреждённая полная строка - ошибка. `recover_combo(directory, remasking, num_steps, seed, run_sha256)` проверяет параметры и хэши всех завершённых задач. Только оборванная последняя строка или хвост следующей незавершённой задачи может быть отброшен; результат с отсутствующим/повреждённым журналом останавливает запуск. Порядок завершённых задач должен быть префиксом `0..99`, повторов быть не должно. `truncate_durable(path, size)` обрезает только уже проверенный незавершённый хвост до байтового offset `size` и делает fsync.

`RUN_METADATA_BASE` содержит конфиг, окружение, source/artifact fingerprints, определение seed и протокол exp1. После GPU-тестов к нему добавится заключение о PyJuice; файл метаданных должен строго совпасть при возобновлении. Перенос между разными GPU/версиями окружения требует нового каталога результатов, чтобы не смешивать условия.

In [ ]:
def combo_paths(directory, remasking, num_steps, seed):
    """directory - каталог; remasking/num_steps/seed определяют пару файлов конфигурации."""
    stem = f"{remasking}_steps{num_steps}_seed{seed}.jsonl"
    return Path(directory) / stem, Path(directory) / f"steplog_{stem}"


def read_jsonl_prefix(path):
    """path - JSONL. Возвращает (список (объект,start,end,raw), offset полных строк, размер файла)."""
    path = Path(path)
    if not path.exists():
        return [], 0, 0
    records = []
    with path.open("rb") as stream:
        while True:
            start = stream.tell()
            raw = stream.readline()
            if not raw:
                return records, start, start
            if not raw.endswith(b"\n"):
                return records, start, stream.tell()
            try:
                obj = json.loads(raw)
            except (ValueError, UnicodeError) as exc:
                raise RuntimeError(f"Повреждена полная строка {path}, offset {start}; автоматического исправления нет.") from exc
            assert isinstance(obj, dict), f"Ожидался объект JSON: {path}:{start}"
            records.append((obj, start, stream.tell(), raw))


def truncate_durable(path, size):
    """path - файл с проверенным незавершённым хвостом; size - последний безопасный byte offset."""
    path = Path(path)
    if path.exists() and path.stat().st_size != size:
        assert 0 <= size <= path.stat().st_size
        print(f"Восстановление {path.name}: отброшен незавершённый хвост {path.stat().st_size - size} байт.")
        with path.open("r+b") as stream:
            stream.truncate(size)
            stream.flush()
            os.fsync(stream.fileno())


def append_durable(path, payload):
    """path - JSONL; payload - готовые bytes с завершающими newline. Одна надёжная допись."""
    with Path(path).open("ab") as stream:
        stream.write(payload)
        stream.flush()
        os.fsync(stream.fileno())


def commit_task(directory, row, step_rows, fail_after=None):
    """directory - хранилище; row/step_rows - одна задача; fail_after - тестовый обрыв trace/result_bytes."""
    assert fail_after in {None, "trace", "result_bytes"}
    assert len(step_rows) == row["num_steps"]
    result_path, trace_path = combo_paths(directory, row["remasking"], row["num_steps"], row["seed"])
    payload = b"".join(canonical_bytes(rec) + b"\n" for rec in step_rows)
    row = dict(row, trace_sha256=hashlib.sha256(payload).hexdigest(), steps_logged=len(step_rows))
    append_durable(trace_path, payload)
    if fail_after == "trace":
        raise InterruptedError("Тестовый обрыв после fsync журнала.")
    row_payload = canonical_bytes(row) + b"\n"
    if fail_after == "result_bytes":
        append_durable(result_path, row_payload[:len(row_payload) // 2])
        raise InterruptedError("Тестовый обрыв в середине строки результата.")
    append_durable(result_path, row_payload)


def recover_combo(directory, remasking, num_steps, seed, run_sha256, expected_gen_length=GEN_LENGTH):
    """directory/правило/шаги/seed - файлы; run_sha256 - идентичность; expected_gen_length - длина, 64 только в smoke."""
    result_path, trace_path = combo_paths(directory, remasking, num_steps, seed)
    results, result_end, _ = read_jsonl_prefix(result_path)
    traces, _, _ = read_jsonl_prefix(trace_path)
    rows = [item[0] for item in results]
    assert [row["task_id"] for row in rows] == list(range(len(rows))), "Повтор, пропуск или неверный порядок task_id."
    assert len(rows) <= N_TASKS
    required = {"task_id", "seed", "task_seed", "correct", "prediction", "extracted_answer",
                "num_steps", "remasking", "pc_active_steps", "generation_ids", "run_sha256",
                "trace_sha256", "steps_logged", "gen_length", "prompt_tokens", "sec",
                "correct_strict", "delta_words", "delta_order"}
    cursor, trace_end = 0, 0
    for row in rows:
        assert required <= set(row), "Неполная строка результата."
        assert (row["remasking"], row["num_steps"], row["seed"]) == (remasking, num_steps, seed)
        assert row["run_sha256"] == run_sha256 and row["gen_length"] == expected_gen_length
        assert row["task_seed"] == task_seed(seed, row["task_id"])
        assert row["prompt_tokens"] == len(PROMPT_IDS[row["task_id"]])
        assert row["correct"] in (0, 1) and row["correct_strict"] in (0, 1)
        assert isinstance(row["prediction"], str) and isinstance(row["extracted_answer"], str)
        assert np.isfinite(row["sec"]) and row["sec"] >= 0
        group = traces[cursor:cursor + num_steps]
        assert len(group) == num_steps and row["steps_logged"] == num_steps, "Завершённой задаче не хватает журнала."
        records = [item[0] for item in group]
        assert all((rec["task_id"], rec["seed"], rec["num_steps"], rec["remasking"], rec["run_sha256"])
                   == (row["task_id"], seed, num_steps, remasking, run_sha256) for rec in records)
        assert hashlib.sha256(b"".join(item[3] for item in group)).hexdigest() == row["trace_sha256"], "Изменились байты журнала."
        validate_trace(records, row["generation_ids"], row["prompt_tokens"], num_steps, expected_gen_length, remasking)
        assert row["pc_active_steps"] == sum(int(rec["pc_active"]) for rec in records)
        assert row["delta_words"] == float(np.mean([rec["delta_words"] for rec in records]))
        assert row["delta_order"] == float(np.mean([rec["delta_order"] for rec in records]))
        cumulative = 0
        for rec in records:
            cumulative += int(rec["pc_active"])
            assert rec["pc_active_steps"] == cumulative
        trace_end = group[-1][2]
        cursor += num_steps
    # Любые лишние полные строки должны относиться только к следующей незавершённой задаче.
    tail = traces[cursor:]
    assert all(item[0].get("task_id") == len(rows) and item[0].get("seed") == seed
               and item[0].get("remasking") == remasking and item[0].get("num_steps") == num_steps
               and item[0].get("run_sha256") == run_sha256 for item in tail), "Конфликтующий хвост журнала."
    assert len(tail) <= num_steps, "В хвосте больше одной незавершённой задачи."
    assert [item[0]["global_step"] for item in tail] == list(range(len(tail)))
    # Мутации только после всех проверок завершённой части.
    truncate_durable(result_path, result_end)
    truncate_durable(trace_path, trace_end)
    return rows


RUN_METADATA_BASE = {
    "schema_version": SCHEMA_VERSION, "experiment": "exp3_hmm_words_only",
    "common_params": COMMON_PARAMS, "common_sha256": COMMON_SHA256,
    "parameters": {"MODEL": MODEL, "HMM_CKPT": HMM_CKPT, "DATASET": DATASET,
                   "DATASET_SPLIT": SPLIT, "N_TASKS": N_TASKS, "GEN_LENGTH": GEN_LENGTH,
                   "BLOCK_LENGTH": BLOCK_LENGTH, "TEMPERATURE": TEMPERATURE,
                   "NUM_STEPS_LIST": NUM_STEPS_LIST, "REMASKING_LIST": REMASKING_LIST,
                   "PC_TEMPERATURE": PC_TEMPERATURE, "PC_FRAC": PC_FRAC,
                   "REVERSE_FRAC": REVERSE_FRAC, "SEEDS": SEEDS,
                   "DTYPE": PRECISION, "ATTN_IMPLEMENTATION": "eager",
                   "CFG_SCALE": CFG_SCALE, "BATCH_SIZE": BATCH_SIZE,
                   "NUM_FEWSHOT": NUM_FEWSHOT, "SYSTEM_INSTRUCTION": SYSTEM_INSTRUCTION,
                   "CAND_FROM": CAND_FROM, "ORDER_FROM": ORDER_FROM},
    "environment": ENVIRONMENT, "artifacts": ARTIFACTS,
    "ids_sha256": IDS_SHA256, "questions_sha256": QUESTIONS_SHA256,
    "documents_sha256": TASKS_DOC["documents_sha256"],
    "prompts_sha256": hashlib.sha256(canonical_bytes(TASKS_DOC["tasks"])).hexdigest(),
    "prediction_definition": "prediction=generation text; extracted_answer=flexible-extract; unlike exp1 prediction",
    "seed_definition": "int.from_bytes(SHA256(UTF8('exp3:{seed}:{task_id}'))[:4], 'little')",
    "reference_notebook_sha256_normalized": REFERENCE_NOTEBOOK_SHA256,
    "reference_protocol": BASELINE_PROTOCOL_CHECK,
    "user_parameter_override": "После сверки с exp1: GEN_LENGTH=512 и HMM_CKPT=il18/llada-gsm-pc",
    "bootstrap": {"repeats": BOOTSTRAP_REPEATS, "seed": BOOTSTRAP_SEED, "confidence": CONFIDENCE_LEVEL},
}
print("Функции определены. Полный прогон ещё заблокирован до секции 5.")

# 5. Дымовой тест на GPU - перед полным прогоном

**Эта секция НЕ выполнялась при подготовке.** На GPU она проверяет синтетические logits, три реальные задачи и оба правила при `SMOKE_GEN_LENGTH=64`, `SMOKE_NUM_STEPS=16`, блоке 32. Эти сокращённые параметры только для smoke и не меняют основной протокол 512/[64,128].

Вспомогательные функции теста:
- `count_backend_calls()` без аргументов считает реальные вызовы трансформера и `juice.queries.sample` через временные hooks/patches. Файлы репозитория не меняются.
- `sample_tape(mode, tape=None)` записывает (`mode="record"`) либо воспроизводит (`"replay"`) только результаты условного выбора скрытых состояний. `tape` - запись из эталонного прогона. При replay проверяется SHA-256 внешних оценок на каждом вызове. Это **только тест**, основной прогон всегда вызывает настоящую HMM.
- `tensor_digest(tensor)` хэширует значения, dtype и форму tensor для сравнения без сохранения больших logits.
- `fixed_state_reproducibility()` сравнивает реальный выбор скрытых состояний на трёх фиксированных состояниях: пять повторов при восстановленных Python/NumPy/PyTorch seeds, при этом исходные logits должны совпасть. Результат - наблюдаемая повторяемость, не математическая гарантия для всех GPU/состояний.
- `smoke_copy(prompt, rule, seed_value, cand_from, order_from, use_pc, collect_trace)` запускает копию на коротком ответе; `prompt` - GPU-токены, `rule` - формула уверенности, `seed_value` - производный seed, остальные параметры соответствуют генератору.

Сначала повторные baseline-прогоны должны совпасть друг с другом. Затем `base/base` без HMM сравнивается с оригиналом. Режим `pc/pc` сравнивается с исходным CoDD при тех же скрытых сэмплах; если PyJuice повторяем по seed, дополнительно сравнивается независимый запуск при том же seed. Прогон с записью/без записи диагностики должен давать одинаковые токены и число вызовов. Детерминированный replay позволяет не выдавать ошибку копии за случайность HMM.

Допустимое ограничение: если на фиксированном входе меняются **сами скрытые сэмплы** при одинаковых logits и восстановленных seeds, а replay-сравнение проходит, разрешено продолжить с явной пометкой: точное воспроизведение после обрыва НЕ ПРОВЕРЕНО/не гарантируется. Любое другое несовпадение останавливает запуск. При неактивной HMM логиты должны совпадать на том же состоянии; сравнивать целые уже разошедшиеся траектории как будто их контекст одинаков - неверно.

In [ ]:
def tensor_digest(tensor):
    """tensor - tensor любых dtype/формы; хэширует его bytes и метаданные (только диагностика smoke)."""
    cpu = tensor.detach().contiguous().cpu()
    payload = cpu.view(torch.uint8).numpy().tobytes()
    header = canonical_bytes({"shape": list(cpu.shape), "dtype": str(cpu.dtype)})
    return hashlib.sha256(header + payload).hexdigest()


@contextmanager
def count_backend_calls():
    """Без аргументов: временно считает фактические forward трансформера и conditional sample HMM."""
    counts = {"transformer": 0, "sample": 0}
    def before_transformer(module, inputs):
        counts["transformer"] += 1
    original_sample = juice.queries.sample
    def counted_sample(*args, **kwargs):
        assert kwargs.get("conditional") is True
        counts["sample"] += 1
        return original_sample(*args, **kwargs)
    handle = model.model.register_forward_pre_hook(before_transformer)
    try:
        with patch.object(juice.queries, "sample", side_effect=counted_sample):
            yield counts
    finally:
        handle.remove()


@contextmanager
def sample_tape(mode, tape=None):
    """mode - record/replay; tape - список записанных скрытых сэмплов; patch действует только внутри теста."""
    assert mode in {"record", "replay"}
    entries = [] if tape is None else tape
    cursor = 0
    original_sample = juice.queries.sample
    def taped_sample(*args, **kwargs):
        nonlocal cursor
        evidence = kwargs["external_categorical_logps"]
        signature = tensor_digest(evidence)
        if mode == "record":
            nodes = original_sample(*args, **kwargs)
            entries.append({"signature": signature, "nodes": nodes.detach().cpu().clone()})
        else:
            assert cursor < len(entries), "Копия вызвала sample лишний раз."
            entry = entries[cursor]
            assert signature == entry["signature"], "При одинаковых скрытых сэмплах разошлись внешние logits HMM."
            nodes = entry["nodes"].to(evidence.device).clone()
        cursor += 1
        return nodes
    with patch.object(juice.queries, "sample", side_effect=taped_sample):
        yield entries
    if mode == "replay":
        assert cursor == len(entries), "Копия пропустила sample эталонного прогона."


def test_synthetic_confidence():
    """Без аргументов: GPU-фикстуры проверяют кандидатов, 4 формулы, маскирование и две delta-метрики."""
    base = torch.tensor([[[8., 0., 0.], [0., 4., 0.], [0., 0., 1.]]], device=DEVICE)
    pc = torch.tensor([[[0., 8., 0.], [0., 0., 1.], [8., 0., 0.]]], device=DEVICE)
    candidates = pc.argmax(-1)
    assert candidates.tolist() == [[1, 2, 0]] and base.argmax(-1).tolist() == [[0, 1, 2]]
    low = confidence_scores(base, candidates, "low_confidence")
    expected = torch.softmax(base.double(), -1).gather(-1, candidates.unsqueeze(-1)).squeeze(-1)
    assert torch.equal(low, expected)
    mask = torch.ones((1, 3), dtype=torch.bool, device=DEVICE)
    assert select_positions(low, mask, 3, 1).tolist() == [2]
    own_low = confidence_scores(base, base.argmax(-1), "low_confidence")
    assert select_positions(own_low, mask, 3, 1).tolist() == [0], "Тест должен выявлять связь low_confidence с кандидатом."
    probs = torch.softmax(base, -1)
    top = probs.topk(2, -1).values
    assert torch.equal(confidence_scores(base, candidates, "margin"), top[..., 0] - top[..., 1])
    assert torch.equal(confidence_scores(base, candidates, "entropy"), (probs * (probs + 1e-10).log()).sum(-1))
    for rule in ["margin", "entropy"]:
        assert torch.equal(confidence_scores(base, candidates, rule),
                           confidence_scores(base, base.argmax(-1), rule))
    shared = torch.tensor([[0.1, 0.9, 0.2]], device=DEVICE)
    cpu_state, gpu_state = torch.get_rng_state().clone(), torch.cuda.get_rng_state().clone()
    assert confidence_scores(base, candidates, "random", shared) is shared
    assert confidence_scores(pc, candidates, "random", shared) is shared
    assert torch.equal(cpu_state, torch.get_rng_state()) and torch.equal(gpu_state, torch.cuda.get_rng_state())
    mask[0, 1] = False
    assert select_positions(shared, mask, 2, 1).tolist() == [0], "Нельзя раскрыть открытую/будущую позицию."
    assert select_positions(shared, mask, 3, 0).numel() == 0
    assert order_distance([], []) == 0.0
    assert order_distance([0, 1], [0, 2]) == 1.0 - 1.0 / 3.0
    assert float((base.argmax(-1) != pc.argmax(-1)).float().mean()) == 1.0
    # Проверка источников не только равных logits: base/base и pc/base обязаны дать разных кандидатов.
    assert not torch.equal(base.argmax(-1), candidates)
    return {"status": "PASSED", "device": str(base.device)}


@torch.no_grad()
def fixed_state_reproducibility():
    """Без аргументов: 3 реальных контекста, 5 повторов реальной HMM, сравнение base logits и скрытых состояний."""
    details = []
    for task_id in range(SMOKE_TASKS):
        prompt = torch.tensor([PROMPT_IDS[task_id]], device=DEVICE, dtype=torch.long)
        # Заполненные позиции блока - реальные токены из данного документа; состояние фиксировано.
        known = lm.tokenizer.encode(INSTANCES[task_id].doc["answer"], add_special_tokens=False)
        assert len(known) >= 18 and model.codd_config.mask_id not in known[:18]
        x = torch.full((1, prompt.shape[1] + BLOCK_LENGTH), model.codd_config.mask_id,
                       device=DEVICE, dtype=torch.long)
        x[:, :prompt.shape[1]] = prompt
        x[0, prompt.shape[1]:prompt.shape[1] + 18] = torch.tensor(known[:18], device=DEVICE)
        sample_hashes, pc_hashes, base_hashes = [], [], []
        for repeat in range(5):
            seed_all(task_seed(0, task_id))
            model._pc_compiled = True  # одинаково прогретое состояние, запись CUDA graph в этом тесте не нужна
            with sample_tape("record") as tape:
                with count_backend_calls() as counts:
                    output = model(x, pc_block_range=(prompt.shape[1], x.shape[1]), use_pc=True,
                                   pc_temperature=PC_TEMPERATURE, pc_frac=PC_FRAC,
                                   reverse_frac=REVERSE_FRAC, return_base_logits=True)
            assert output.pc_active and len(tape) == 1
            assert counts == {"transformer": 1, "sample": 1}
            base_hashes.append(tensor_digest(output.logits_base))
            pc_hashes.append(tensor_digest(output.logits))
            sample_hashes.append(tensor_digest(tape[0]["nodes"]))
            # Подкласс и исходный forward сравниваются на одном состоянии и одном скрытом сэмпле.
            seed_all(task_seed(0, task_id))
            model._pc_compiled = True
            with sample_tape("replay", tape):
                expected = model(x, pc_block_range=(prompt.shape[1], x.shape[1]), use_pc=True,
                                 pc_temperature=PC_TEMPERATURE, pc_frac=PC_FRAC,
                                 reverse_frac=REVERSE_FRAC)
            assert torch.equal(output.logits, expected.logits), "Расхождение forward при фиксированном сэмпле."
            del output, expected, tape
        assert len(set(base_hashes)) == 1, "Трансформер недетерминирован на фиксированном состоянии."
        if len(set(sample_hashes)) == 1:
            assert len(set(pc_hashes)) == 1, "PC меняет logits при одинаковых скрытых состояниях."
        details.append({"task_id": task_id, "repeats": 5,
                        "unique_hidden_samples": len(set(sample_hashes)),
                        "unique_pc_outputs": len(set(pc_hashes)),
                        "base_logits_equal": True})
    return all(item["unique_hidden_samples"] == 1 for item in details), details


def smoke_copy(prompt, rule, seed_value, cand_from, order_from, use_pc, collect_trace=True):
    """prompt - GPU tensor; rule/seed_value - правило/seed; источники/use_pc/collect_trace - параметры копии."""
    seed_all(seed_value)
    model._pc_compiled = False
    with count_backend_calls() as counts:
        result = llada_words_only_generate(
            model, prompt, num_steps=SMOKE_NUM_STEPS, gen_length=SMOKE_GEN_LENGTH,
            block_length=BLOCK_LENGTH, temperature=TEMPERATURE, cfg_scale=CFG_SCALE,
            remasking=rule, mask_id=model.codd_config.mask_id, use_pc=use_pc,
            pc_temperature=PC_TEMPERATURE, pc_frac=PC_FRAC, reverse_frac=REVERSE_FRAC,
            cand_from=cand_from, order_from=order_from, collect_trace=collect_trace)
    assert counts == {"transformer": SMOKE_NUM_STEPS, "sample": result.pc_active_steps}
    if collect_trace:
        validate_trace(result.trace, result.tokens[0, prompt.shape[1]:].tolist(), prompt.shape[1],
                       SMOKE_NUM_STEPS, SMOKE_GEN_LENGTH, rule, use_pc=use_pc)
    return result, counts


def smoke_original(prompt, rule, seed_value, use_pc):
    """prompt - GPU tensor; rule - уверенность; seed_value - восстановленный seed; use_pc - исходный режим HMM."""
    seed_all(seed_value)
    model._pc_compiled = False
    with count_backend_calls() as counts:
        tokens = llada_diffusion_generate(
            model, prompt, num_steps=SMOKE_NUM_STEPS, gen_length=SMOKE_GEN_LENGTH,
            block_length=BLOCK_LENGTH, temperature=TEMPERATURE, cfg_scale=CFG_SCALE,
            remasking=rule, mask_id=model.codd_config.mask_id, use_pc=use_pc,
            pc_temperature=PC_TEMPERATURE, pc_frac=PC_FRAC, reverse_frac=REVERSE_FRAC)
    assert counts["transformer"] == SMOKE_NUM_STEPS
    assert not bool((tokens[:, prompt.shape[1]:] == model.codd_config.mask_id).any())
    assert torch.equal(tokens[:, :prompt.shape[1]], prompt)
    if not use_pc:
        assert counts["sample"] == 0
    return tokens, counts

## 5.1. Возобновление на реальных smoke-ответах

`smoke_record(task_id, rule, seed, result)` упаковывает короткий результат в ту же схему, что основной прогон, но явно с `gen_length=64` и тестовым run fingerprint. `test_resume(records)` получает две такие задачи и в отдельном временном каталоге проверяет: непрерывную запись, обрыв после журнала, обрыв внутри результата, повторный запуск без дубликатов и отказ при конфликтующих метаданных. Завершённые байты первой задачи должны сохраниться, итоговые файлы - совпасть с непрерывной записью. Секунды здесь фиксированы в **тестовых фикстурах** и не являются измерением производительности.

Повторная генерация с той же task seed проверяется отдельно в smoke с включённым/выключенным журналом. Если PyJuice не повторяем, файловое восстановление всё равно проверяется, но точное восстановление токенов повторно сгенерированной незавершённой задачи не обещается.

In [ ]:
def smoke_record(task_id, rule, seed, result):
    """task_id/rule/seed - smoke-конфигурация; result - реальный GenerationResult; фикстуры не попадают в полный прогон."""
    prompt_length = len(PROMPT_IDS[task_id])
    ids = result.tokens[0, prompt_length:].tolist()
    text = postprocess_generated_text(lm.tok_decode(ids), stop=UNTIL, think_end_token=None)
    scores, predictions = score_response(INSTANCES[task_id], text)
    run_sha = "smoke-fixture-not-a-production-run"
    row = {"task_id": task_id, "seed": seed, "task_seed": task_seed(seed, task_id),
           "correct": scores["flexible-extract"], "prediction": text,
           "extracted_answer": predictions["flexible-extract"], "generation": text,
           "correct_strict": scores["strict-match"], "prediction_strict": predictions["strict-match"],
           "num_steps": SMOKE_NUM_STEPS, "remasking": rule, "pc_active_steps": result.pc_active_steps,
           "delta_words": float(np.mean([s["delta_words"] for s in result.trace])),
           "delta_order": float(np.mean([s["delta_order"] for s in result.trace])),
           "gen_length": SMOKE_GEN_LENGTH, "prompt_tokens": prompt_length,
           "generation_ids": ids, "sec": 0.0, "run_sha256": run_sha}
    steps, cumulative = [], 0
    for rec in result.trace:
        cumulative += int(rec["pc_active"])
        steps.append({"task_id": task_id, "seed": seed, "task_seed": task_seed(seed, task_id),
                      "remasking": rule, "num_steps": SMOKE_NUM_STEPS,
                      "run_sha256": run_sha, "pc_active_steps": cumulative, **rec})
    return row, steps


def test_resume(records):
    """records - две пары (row, step_rows) реальных smoke-задач 0/1 одной конфигурации."""
    assert len(records) == 2 and [r[0]["task_id"] for r in records] == [0, 1]
    rule, seed = records[0][0]["remasking"], records[0][0]["seed"]
    run_sha = records[0][0]["run_sha256"]
    with tempfile.TemporaryDirectory(prefix=".smoke-resume-", dir=OUT_DIR) as work:
        reference = Path(work) / "uninterrupted"
        reference.mkdir()
        for row, steps in records:
            commit_task(reference, row, steps)
        expected_files = [path.read_bytes() for path in combo_paths(reference, rule, SMOKE_NUM_STEPS, seed)]
        for failure in ["trace", "result_bytes"]:
            directory = Path(work) / failure
            directory.mkdir()
            commit_task(directory, *records[0])
            first_files = [path.read_bytes() for path in combo_paths(directory, rule, SMOKE_NUM_STEPS, seed)]
            try:
                commit_task(directory, *records[1], fail_after=failure)
            except InterruptedError:
                pass
            else:
                raise AssertionError("Тестовый обрыв не сработал.")
            restored = recover_combo(directory, rule, SMOKE_NUM_STEPS, seed, run_sha,
                                     expected_gen_length=SMOKE_GEN_LENGTH)
            assert len(restored) == 1
            assert [path.read_bytes() for path in combo_paths(directory, rule, SMOKE_NUM_STEPS, seed)] == first_files
            commit_task(directory, *records[1])
            for _ in range(2):
                restored = recover_combo(directory, rule, SMOKE_NUM_STEPS, seed, run_sha,
                                         expected_gen_length=SMOKE_GEN_LENGTH)
                assert [r["task_id"] for r in restored] == [0, 1]
            paths = combo_paths(directory, rule, SMOKE_NUM_STEPS, seed)
            assert [path.read_bytes() for path in paths] == expected_files
            before_conflict = [path.read_bytes() for path in paths]
            try:
                recover_combo(directory, rule, SMOKE_NUM_STEPS, seed, "wrong-run",
                              expected_gen_length=SMOKE_GEN_LENGTH)
            except AssertionError:
                pass
            else:
                raise AssertionError("Конфликт метаданных принят.")
            assert [path.read_bytes() for path in paths] == before_conflict
            # Полная повреждённая строка в журнале не должна считаться оборванным хвостом.
            append_durable(paths[1], b"not-json\n")
            try:
                recover_combo(directory, rule, SMOKE_NUM_STEPS, seed, run_sha,
                              expected_gen_length=SMOKE_GEN_LENGTH)
            except RuntimeError:
                pass
            else:
                raise AssertionError("Повреждённая полная строка была принята.")
    return {"status": "PASSED", "failures_tested": ["trace", "result_bytes", "metadata", "corrupt_complete_line"]}

In [ ]:
SMOKE_PASSED = False
SMOKE_REPORT = {"status": "RUNNING"}
synthetic_check = test_synthetic_confidence()
PYJUICE_REPRODUCIBLE, fixed_state_check = fixed_state_reproducibility()
if not PYJUICE_REPRODUCIBLE:
    print("ОГРАНИЧЕНИЕ: скрытые состояния PyJuice меняются при восстановленных seeds и одинаковых base logits.")
    print("Результаты HMM проверяются с replay. Точное воспроизведение незавершённой задачи после обрыва не гарантируется.")
smoke_checks, resume_records = [], []
for rule in REMASKING_LIST:
    for task_id in range(SMOKE_TASKS):
        prompt = torch.tensor([PROMPT_IDS[task_id]], device=DEVICE, dtype=torch.long)
        derived = task_seed(0, task_id)
        original_base, _ = smoke_original(prompt, rule, derived, use_pc=False)
        repeated_base, _ = smoke_original(prompt, rule, derived, use_pc=False)
        assert torch.equal(original_base, repeated_base), "Исходная LLaDA недетерминирована."
        copied_base, _ = smoke_copy(prompt, rule, derived, "base", "base", use_pc=False)
        assert torch.equal(original_base, copied_base.tokens), "base/base не совпал с оригиналом."
        with sample_tape("record") as canonical_tape:
            original_pc, original_counts = smoke_original(prompt, rule, derived, use_pc=True)
        with sample_tape("replay", canonical_tape):
            copied_pc, copy_counts = smoke_copy(prompt, rule, derived, "pc", "pc", use_pc=True)
        assert torch.equal(original_pc, copied_pc.tokens), "pc/pc не совпал при одинаковых скрытых состояниях."
        assert original_counts == copy_counts
        assert copied_pc.pc_active_steps > 0, "Smoke не проверил активную HMM."
        if PYJUICE_REPRODUCIBLE:
            independent_pc, _ = smoke_copy(prompt, rule, derived, "pc", "pc", use_pc=True)
            assert torch.equal(original_pc, independent_pc.tokens), (
                "pc/pc не совпал при том же seed, хотя фиксированные тесты были повторяемы. Остановка; причина НЕ ПРОВЕРЕНА.")
            del independent_pc
        with sample_tape("record") as words_tape:
            words_only, words_counts = smoke_copy(prompt, rule, derived, CAND_FROM, ORDER_FROM, use_pc=True)
        with sample_tape("replay", words_tape):
            no_log, no_log_counts = smoke_copy(prompt, rule, derived, CAND_FROM, ORDER_FROM,
                                              use_pc=True, collect_trace=False)
        assert torch.equal(words_only.tokens, no_log.tokens) and words_counts == no_log_counts
        assert no_log.trace == []
        if PYJUICE_REPRODUCIBLE:
            repeated_words, _ = smoke_copy(prompt, rule, derived, CAND_FROM, ORDER_FROM,
                                          use_pc=True, collect_trace=False)
            assert torch.equal(words_only.tokens, repeated_words.tokens), "Task seed не воспроизвёл words-only."
            del repeated_words
        inactive = [rec for rec in words_only.trace if not rec["pc_active"]]
        assert len(inactive) >= 2 and all(rec["delta_words"] == rec["delta_order"] == 0 for rec in inactive)
        if rule == REMASKING_LIST[0] and task_id < 2:
            resume_records.append(smoke_record(task_id, rule, 0, words_only))
        smoke_checks.append({"task_id": task_id, "remasking": rule,
                             "base_tokens_equal": True, "pc_tokens_equal_replay": True,
                             "pc_tokens_equal_seed": True if PYJUICE_REPRODUCIBLE else "НЕ ГАРАНТИРУЕТСЯ",
                             "logging_preserves_tokens_and_calls": True,
                             "counts": words_counts, "inactive_steps_checked": len(inactive)})
        print(f"Smoke OK: task={task_id}, {rule}, transformer={words_counts['transformer']}, sample={words_counts['sample']}")
        del original_base, repeated_base, copied_base, original_pc, copied_pc, words_only, no_log
        del canonical_tape, words_tape
        torch.cuda.empty_cache()
resume_check = test_resume(resume_records)
SMOKE_REPORT = {"status": "PASSED", "synthetic": synthetic_check,
                "fixed_state_hidden_samples": fixed_state_check,
                "pyjuice_repeatable_on_tested_states": PYJUICE_REPRODUCIBLE,
                "generation_checks": smoke_checks, "resume": resume_check,
                "resume_regeneration_equal_on_tested_tasks": PYJUICE_REPRODUCIBLE,
                "gen_length": SMOKE_GEN_LENGTH, "num_steps": SMOKE_NUM_STEPS,
                "full_length_peak_memory": "НЕ ПРОВЕРЕНО: smoke использует сокращённую длину",
                "limitations": [] if PYJUICE_REPRODUCIBLE else [
                    "PyJuice не повторяем при восстановленных seeds; точное восстановление токенов незавершённой задачи не гарантируется."]}
SMOKE_PASSED = True
print("SMOKE_PASSED=True. GPU-проверки пройдены в этом ядре; основной прогон ещё не запускался.")

# 6. Полный прогон

После секции 5 сохраните блокнот с окончательными настройками и установите `RUN_FULL=True` в конфиге. При `False` эта секция только сообщает о пропуске. Не повторяйте ячейку загрузки весов ради переключения флага: можно изменить значение `RUN_FULL` в конфиге и выполнить только эту ячейку, если остальные параметры не менялись.

`run_combo(remasking, num_steps, seed, run_sha256)` восстанавливает одну конфигурацию, пропускает готовый префикс задач и после каждой новой задачи надёжно пишет журнал, затем результат. Значения параметров: формула уверенности, общее число шагов, seed повтора и fingerprint метаданных. Основные источники фиксированы: кандидаты PC, уверенность base. Трансформер/HMM остаются теми же объектами во всех 12 конфигурациях.

`freeze_run_metadata()` без аргументов добавляет результаты проверки seed к метаданным и проверяет существующий файл до возобновления. Отпечаток самого блокнота рассчитывается по исходникам ячеек, без outputs/execution_count: сохранение вывода не меняет идентичность кода. Изменение алгоритма, параметров или окружения запрещает продолжить старые файлы молча.

`replace_json(path, value)` атомарно заменяет служебную сводку/отчёт о smoke; для идентичности запуска используется более строгий `write_json_once`. Пакеты не устанавливаются, новая модель не загружается. GPU latency включает диагностику; это не чистая скорость исходного CoDD. После работы остановите вычислительный ресурс в интерфейсе DataSphere; блокнот сам его не останавливает.

In [ ]:
def replace_json(path, value):
    """path - служебная сводка; value - новые JSON-данные. Атомарная замена, не для файлов результатов задач."""
    path = Path(path)
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                     prefix=f".{path.name}.", delete=False) as stream:
        temporary = Path(stream.name)
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write("\n")
        stream.flush()
        os.fsync(stream.fileno())
    try:
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def freeze_run_metadata():
    """Без аргументов: фиксирует/сверяет код, условия, проверку HMM и возвращает metadata с его SHA-256."""
    assert SMOKE_PASSED and SMOKE_REPORT["status"] == "PASSED"
    metadata = copy.deepcopy(RUN_METADATA_BASE)
    metadata["notebook_source_sha256"] = notebook_source_sha256(REPO_ROOT / NOTEBOOK_RELATIVE_PATH)
    metadata["pyjuice_repeatable_on_tested_states"] = PYJUICE_REPRODUCIBLE
    fingerprint = hashlib.sha256(canonical_bytes(metadata)).hexdigest()
    write_json_once(OUT_DIR / "run_metadata.json", {"run_sha256": fingerprint, **metadata})
    replace_json(OUT_DIR / "smoke.json", SMOKE_REPORT)
    return metadata, fingerprint


def run_combo(remasking, num_steps, seed, run_sha256):
    """remasking - правило; num_steps - бюджет по всем блокам; seed - повтор; run_sha256 - условия запуска."""
    rows = recover_combo(OUT_DIR, remasking, num_steps, seed, run_sha256)
    print(f"{remasking}, steps={num_steps}, seed={seed}: уже готово {len(rows)}/{N_TASKS}")
    for task_id in range(len(rows), N_TASKS):
        row, step_rows = process_task(task_id, remasking, num_steps, seed, run_sha256)
        commit_task(OUT_DIR, row, step_rows)
        print(f"  task={task_id}: correct={row['correct']}, HMM steps={row['pc_active_steps']}, sec={row['sec']:.2f}")
        del row, step_rows
        torch.cuda.empty_cache()
    verified = recover_combo(OUT_DIR, remasking, num_steps, seed, run_sha256)
    assert len(verified) == N_TASKS
    return len(verified)


RUN_SHA256 = None
if RUN_FULL:
    assert SMOKE_PASSED, "Сначала успешно выполните секцию 5 в этом ядре."
    RUN_METADATA, RUN_SHA256 = freeze_run_metadata()
    for rule in REMASKING_LIST:
        for steps in NUM_STEPS_LIST:
            for seed in SEEDS:
                run_combo(rule, steps, seed, RUN_SHA256)
    print("Полный прогон завершён: 12 конфигураций x 100 задач.")
else:
    print("RUN_FULL=False: полный прогон пропущен. Никакие результаты не придуманы.")

# 7. Сводка и отчёт

`bootstrap_ci(values)` принимает одномерный массив точностей по задачам. Выполняет 2000 percentile-bootstrap повторов с независимым генератором `BOOTSTRAP_SEED=20261023`, возвращает среднее и 95% границы; RNG генерации не меняется. Для каждой конфигурации сначала показываются три seed отдельно. Для общего интервала усредняем три значения одной задачи и ресемплируем **задачи целыми группами из трёх seed**, не считаем 300 наблюдений независимыми.

`build_summary(run_sha256)` читает проверенные файлы этого эксперимента; `run_sha256` должен совпасть с сохранёнными условиями. Пропуски помечаются как неполный прогон. Общая оценка и разброс по seeds публикуются только при наличии всех 100 задач у всех трёх seeds. `accuracy_seed_std` - выборочное стандартное отклонение трёх seed-точностей, не стандартная ошибка и не bootstrap CI. Latency - секунды генерации с логированием, включая синхронизации/диагностику, без загрузки/оценки/записи.

Отчёт объясняет все изменения hmm-words-only, настройки, статус проверок и ограничения. Показатели `low_confidence` и `margin` сравниваются только внутри exp3; вывод о пользе HMM относительно baseline здесь не делается. Совпадение промптов с exp1 - проверка протокола, не сравнение качества.

In [ ]:
def bootstrap_ci(values):
    """values - непустой одномерный массив по задачам; конфиг задаёт повторы, seed и уровень CI."""
    values = np.asarray(values, dtype=np.float64)
    assert values.ndim == 1 and values.size > 0 and np.isfinite(values).all()
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    indices = rng.integers(0, values.size, size=(BOOTSTRAP_REPEATS, values.size))
    means = values[indices].mean(axis=1)
    alpha = 1.0 - CONFIDENCE_LEVEL
    low, high = np.quantile(means, [alpha / 2, 1.0 - alpha / 2])
    return float(values.mean()), float(low), float(high)


def build_summary(run_sha256):
    """run_sha256 - идентичность условий; возвращает per_seed/grouped/complete для файлов только exp3."""
    per_seed, grouped = [], []
    for rule in REMASKING_LIST:
        for steps in NUM_STEPS_LIST:
            seed_results = {}
            for seed in SEEDS:
                rows = recover_combo(OUT_DIR, rule, steps, seed, run_sha256)
                seed_results[seed] = rows
                if not rows:
                    per_seed.append({"remasking": rule, "num_steps": steps, "seed": seed,
                                     "n_tasks": 0, "complete": False, "status": "НЕ ПРОВЕРЕНО: нет результатов"})
                    continue
                accuracy, low, high = bootstrap_ci([row["correct"] for row in rows])
                strict_accuracy, strict_low, strict_high = bootstrap_ci([row["correct_strict"] for row in rows])
                per_seed.append({"remasking": rule, "num_steps": steps, "seed": seed,
                                 "n_tasks": len(rows), "complete": len(rows) == N_TASKS,
                                 "accuracy": accuracy, "ci_low": low, "ci_high": high,
                                 "accuracy_strict": strict_accuracy, "ci_low_strict": strict_low,
                                 "ci_high_strict": strict_high,
                                 "pc_active_steps_mean": float(np.mean([r["pc_active_steps"] for r in rows])),
                                 "delta_words_mean": float(np.mean([r["delta_words"] for r in rows])),
                                 "delta_order_mean": float(np.mean([r["delta_order"] for r in rows])),
                                 "latency_mean_sec": float(np.mean([r["sec"] for r in rows])),
                                 "latency_median_sec": float(np.median([r["sec"] for r in rows]))})
            complete = all(len(seed_results[seed]) == N_TASKS for seed in SEEDS)
            if not complete:
                grouped.append({"remasking": rule, "num_steps": steps, "complete": False,
                                "status": "НЕПОЛНЫЙ ПРОГОН: общая оценка по трём seed не рассчитана"})
                continue
            matrix = np.asarray([[row["correct"] for row in seed_results[seed]] for seed in SEEDS], dtype=float)
            assert matrix.shape == (len(SEEDS), N_TASKS)
            accuracy, low, high = bootstrap_ci(matrix.mean(axis=0))
            all_rows = [row for seed in SEEDS for row in seed_results[seed]]
            grouped.append({"remasking": rule, "num_steps": steps, "complete": True,
                            "n_tasks": N_TASKS, "seeds": SEEDS,
                            "accuracy_mean": accuracy, "accuracy_seed_std": float(matrix.mean(axis=1).std(ddof=1)),
                            "ci_low": low, "ci_high": high,
                            "pc_active_steps_mean": float(np.mean([r["pc_active_steps"] for r in all_rows])),
                            "delta_words_mean": float(np.mean([r["delta_words"] for r in all_rows])),
                            "delta_order_mean": float(np.mean([r["delta_order"] for r in all_rows])),
                            "latency_mean_sec": float(np.mean([r["sec"] for r in all_rows])),
                            "latency_median_sec": float(np.median([r["sec"] for r in all_rows]))})
    return {"experiment": "exp3_hmm_words_only", "run_sha256": run_sha256,
            "gen_length": GEN_LENGTH, "hmm_ckpt": HMM_CKPT, "ids_sha256": IDS_SHA256,
            "bootstrap_repeats": BOOTSTRAP_REPEATS, "bootstrap_seed": BOOTSTRAP_SEED,
            "confidence_level": CONFIDENCE_LEVEL,
            "per_seed": per_seed, "grouped": grouped,
            "complete": len(grouped) == 4 and all(row["complete"] for row in grouped)}


SUMMARY = None
if (OUT_DIR / "run_metadata.json").exists():
    assert SMOKE_PASSED, "Для проверки сохранённых результатов сначала выполните smoke в этом ядре."
    # freeze не разрешает анализировать старые результаты как будто они получены с новым кодом/окружением.
    RUN_METADATA, RUN_SHA256 = freeze_run_metadata()
    SUMMARY = build_summary(RUN_SHA256)
    display(pd.DataFrame(SUMMARY["per_seed"]))
    display(pd.DataFrame(SUMMARY["grouped"]))
    replace_json(OUT_DIR / "summary.json", SUMMARY)
    print("Полный прогон:", SUMMARY["complete"])
else:
    print("Точность, CI и latency НЕ ПРОВЕРЕНЫ: основной прогон не запускался.")

## 7.1. Объяснение реализации и границ выводов

Что изменено для **hmm-words-only**:
1. Добавлен `return_base_logits` в подкласс модели: без него эталонный путь оригинален; с ним исходные logits сохраняются до in-place добавки HMM. Это позволяет не запускать второй трансформер и не семплировать вторую цепочку скрытых состояний для сравнения.
2. В копии генератора разделены `cand_from` (какие token IDs предложить) и `order_from` (по каким logits вычислить уверенность). Основной режим `pc/base` меняет только эти источники; расписание, ограничения позиции и top-k сохранены. Температура 0 означает argmax токенов после **совместного сэмпла скрытой цепочки**, а не прямой сэмпл токенов из полного совместного распределения и не argmax одномерных маргиналов CoDD.
3. `low_confidence` использует вероятность выбранного CoDD-токена под LLaDA: это не полностью фиксированный порядок исходной LLaDA. `margin` использует разность двух максимальных вероятностей LLaDA; с текущим кандидатом напрямую не связан. Оба правила обязательны. Сравнение их точностей описывает поведение правил внутри exp3, а не автоматически причинный вклад HMM.
4. Добавлены диагностики на одном состоянии: `delta_words` измеряет изменение argmax среди масок блока; `delta_order` - несовпадение наборов раскрываемых позиций при одинаковых кандидатах. Когда HMM неактивна, требуются равные logits и нулевые delta. Усреднение delta по всем шагам включает неактивные нули.
5. Обязательная HMM, сохранённые source/model/data fingerprints, производные seeds, fsync и контрольная сумма журнала нужны для проверки корректности и продолжения работы DataSphere. Заглушки PC, молчаливая замена dtype, обрезка промптов и автоматическое изменение порога/температуры запрещены.
6. Использованы запросы/токенизация/stop-обработка/фильтры/метрика из того же harness, что в exp1. По решению пользователя длина **512**, HMM **gsm-PC**; это явное изменение относительно исходного задания 256/math-PC. Порог 0.5 применяется строго по доле оставшихся масок в блоке, не означает ровно половину шагов. Температура PC 0.2 задаёт заострение оценок перед условным выводом HMM; это не температура семплирования токенов, которая равна 0.

Допущения: один процесс, одна NVIDIA GPU с аппаратным bfloat16, один ответ за раз, 100 первых задач, zero-shot, CFG=0, блок 32, бюджет 64/128 шагов по всему ответу. Совместимость API PyJuice/памяти/драйвера подтверждается только GPU-запуском. Наблюдаемое повторение пяти сэмплов на трёх состояниях не гарантирует детерминизм на всех состояниях. Если PyJuice неподконтролен seed, готовые записи сохраняются точно, но повторный расчёт незавершённой задачи может дать другой ответ.

Без полного прогона никаких выводов о точности/скорости не делается. Другие эксперименты и таблицы статьи здесь не оцениваются.

In [ ]:
report_lines = [
    "# Отчёт: exp3_hmm_words_only",
    "",
    f"Параметры: {MODEL}; HMM={HMM_CKPT}; длина={GEN_LENGTH}; блок={BLOCK_LENGTH}; "
    f"шаги={NUM_STEPS_LIST}; правила={REMASKING_LIST}; seeds={SEEDS}.",
    "Кандидаты: pc; уверенность: base; token temperature=0; PC temperature=0.2; mask-ratio threshold<0.5; CFG=0.",
    "По решению пользователя после сверки с exp1 исходные 256/math-PC заменены на 512/gsm-PC.",
    "",
    "Реализация: один forward трансформера, клон до in-place PC, один совместный скрытый сэмпл на активный шаг; "
    "раздельные источники кандидатов/оценок и неизменённые формулы/top-k. Подробное объяснение каждого изменения "
    "и параметров функций находится в разделах 2, 4, 5 и 7.1 блокнота.",
    "low_confidence остаётся связанным с кандидатом CoDD; margin на том же состоянии от кандидата не зависит.",
    "",
    f"GPU smoke: {SMOKE_REPORT.get('status', 'НЕ ПРОВЕРЕНО')}.",
    f"Повторяемость скрытых сэмплов на проверенных состояниях: {PYJUICE_REPRODUCIBLE}.",
    f"Сопоставимость протокола exp1: {BASELINE_PROTOCOL_CHECK}.",
    f"Хэш ID задач: {IDS_SHA256}.",
]
if SUMMARY is None:
    report_lines += ["", "Точность, доверительные интервалы и latency: НЕ ПРОВЕРЕНО - нет основного прогона."]
else:
    report_lines += ["", f"Полнота 12 конфигураций: {SUMMARY['complete']}.",
                     "Bootstrap: 2000 повторов, 95%, отдельный RNG; в общей оценке seed группируются по задаче."]
    completed = {(r["remasking"], r["num_steps"]): r for r in SUMMARY["grouped"] if r["complete"]}
    for steps in NUM_STEPS_LIST:
        left, right = completed.get(("low_confidence", steps)), completed.get(("margin", steps))
        if left is not None and right is not None:
            difference = 100 * (right["accuracy_mean"] - left["accuracy_mean"])
            report_lines.append(f"{steps} шагов: low_confidence={100 * left['accuracy_mean']:.2f}%, "
                                f"margin={100 * right['accuracy_mean']:.2f}%; разность margin-low_confidence="
                                f"{difference:+.2f} п.п. Это наблюдение внутри exp3, не доказательство пользы HMM.")
        else:
            report_lines.append(f"{steps} шагов: сравнение правил НЕ ПРОВЕРЕНО - неполные результаты.")
    report_lines += ["Подробные интервалы, разброс seed, PC activity, delta-метрики и latency - в summary.json.",
                     "Latency включает диагностику: напрямую сравнивать с чистой скоростью исходного CoDD нельзя."]
report_lines += ["", "Ограничения: первые 100 задач; выводы не обобщаются автоматически на весь GSM8K. "
                 "Сравнения качества с другими экспериментами/статьёй не выполнялись."]
report_lines.extend(SMOKE_REPORT.get("limitations", []))
REPORT_SUMMARY = "\n".join(report_lines) + "\n"
# Экспортируем самостоятельный отчёт с уже написанными пояснениями и параметрами функций.
report_notebook = json.loads((REPO_ROOT / NOTEBOOK_RELATIVE_PATH).read_text(encoding="utf-8"))
explanation_ids = {
    "exp3-model-guide", "exp3-generation-guide", "exp3-scoring-guide", "exp3-cache-guide",
    "exp3-smoke-guide", "exp3-resume-test-guide", "exp3-full-guide", "exp3-summary-guide",
    "exp3-report-guide"}
explanations = ["".join(cell["source"]) for cell in report_notebook["cells"]
                if cell["id"] in explanation_ids]
assert len(explanations) == len(explanation_ids)
REPORT_TEXT = REPORT_SUMMARY + "\n## Подробное объяснение изменений и параметров\n\n" + "\n\n".join(explanations) + "\n"
display(Markdown(REPORT_SUMMARY))
if SUMMARY is not None:
    # Служебный отчёт заменяется атомарно; файлы задач остаются append-only.
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=OUT_DIR,
                                     prefix=".report.", delete=False) as stream:
        report_tmp = Path(stream.name)
        stream.write(REPORT_TEXT)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(report_tmp, OUT_DIR / "report.md")
print("После завершения остановите вычислительный ресурс в интерфейсе DataSphere.")